# Backend Overview

This notebook is a **read-only copy of the final V2 backend**.

It is for the viva code explanation only.

The live demo is a different notebook: `V2/notebooks/colab_phase21_final_live_demo.ipynb`.

Do **not** use this notebook to rerun the 420-case benchmark, rebuild the knowledge base, recalibrate T, or rerun the judge.

The code cells below are the **exact files** from the repository.

| Part | What it is |
|---|---|
| Retrieval backend | Get PDFs, extract text, chunk, embed, store in Chroma, retrieve |
| RAG backend | Three architectures: Single-Agent, Multi-Agent, Multi-Agent + UQ |
| Prompt backend | The actual prompts sent to Qwen |
| Calibration & threshold | How T = 0.65 was chosen on DEV 40 and locked |
| Evaluation & metrics | Numeric correctness, coverage, judge faithfulness, and the other scores |
| Statistical analysis | McNemar, Spearman, Mann–Whitney, Holm, effect sizes |
| Streamlit | The live artefact UI |
| RQ connection | Which file supports RQ1, RQ2 and RQ3 |


# 1. Retrieval Backend


## `pdf_fetch.py`

**File:** `V2/src/retrieval/pdf_fetch.py`

**What this file does**  
Collects which FinQA page PDFs belong in the knowledge base, then downloads them from Hugging Face.

**What goes in**  
The frozen TEST CSV, the frozen DEV calibration CSV, and how many extra train PDFs to add as distractors (50).

**What comes out**  
A list of documents (`CorpusDoc`) and saved PDF files under `knowledge_base/documents/`.

**What I can say in the viva**  
This is where documents are obtained. The path rule is `data/FinQA/{split}/{file_name}`. Gold answers are not downloaded as documents.


In [ ]:
"""Download FinQA page PDFs from the Hugging Face dataset repo."""

from __future__ import annotations

import csv
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

from huggingface_hub import hf_hub_download


@dataclass(frozen=True)
class CorpusDoc:
    split: str
    file_name: str
    role: str  # test | calibration | distractor
    company_symbol: str = ""
    company_name: str = ""
    report_year: str = ""
    context_id: str = ""
    question_id: str = ""

    @property
    def repo_path(self) -> str:
        return f"data/FinQA/{self.split}/{self.file_name}"

    @property
    def doc_key(self) -> str:
        return f"{self.split}::{self.file_name}"


def _read_csv_docs(path: Path, *, split: str, role: str) -> list[CorpusDoc]:
    docs: list[CorpusDoc] = []
    with path.open(encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle):
            file_name = str(row.get("file_name") or "").strip()
            if not file_name:
                continue
            docs.append(
                CorpusDoc(
                    split=split,
                    file_name=file_name,
                    role=role,
                    company_symbol=str(row.get("company_symbol") or ""),
                    company_name=str(row.get("company_name") or ""),
                    report_year=str(row.get("report_year") or ""),
                    context_id=str(row.get("context_id") or ""),
                    question_id=str(row.get("id") or ""),
                )
            )
    return docs


def _dedupe_by_doc_key(docs: Iterable[CorpusDoc]) -> list[CorpusDoc]:
    seen: set[str] = set()
    out: list[CorpusDoc] = []
    for doc in docs:
        if doc.doc_key in seen:
            continue
        seen.add(doc.doc_key)
        out.append(doc)
    return out


# Work out which PDFs we need
def collect_corpus_targets(
    *,
    test_csv: Path,
    calibration_csv: Path,
    distractor_count: int = 50,
    distractor_seed: int = 42,
    dataset_id: str = "G4KMU/t2-ragbench",
    subset: str = "FinQA",
) -> list[CorpusDoc]:
    """Collect unique PDFs for test + calibration, plus optional train distractors."""
    core = _dedupe_by_doc_key(
        [
            # TEST + DEV PDFs first
            *_read_csv_docs(test_csv, split="test", role="test"),
            *_read_csv_docs(calibration_csv, split="dev", role="calibration"),
        ]
    )
    core_files = {d.file_name for d in core}
    if distractor_count <= 0:
        return core

    from datasets import load_dataset
    import random

    # Add extra train PDFs
    ds = load_dataset(dataset_id, subset, split="train")
    candidates: list[CorpusDoc] = []
    seen_files: set[str] = set(core_files)
    for row in ds:
        file_name = str(row.get("file_name") or "").strip()
        if not file_name or file_name in seen_files:
            continue
        seen_files.add(file_name)
        candidates.append(
            CorpusDoc(
                split="train",
                file_name=file_name,
                role="distractor",
                company_symbol=str(row.get("company_symbol") or ""),
                company_name=str(row.get("company_name") or ""),
                report_year=str(row.get("report_year") or ""),
                context_id=str(row.get("context_id") or ""),
                question_id=str(row.get("id") or ""),
            )
        )
    rng = random.Random(distractor_seed)
    rng.shuffle(candidates)
    return core + candidates[:distractor_count]


def download_pdfs(
    docs: list[CorpusDoc],
    *,
    documents_dir: Path,
    repo_id: str = "G4KMU/t2-ragbench",
) -> dict:
    """Download page PDFs into ``documents_dir/{split}/{file_name}``."""
    documents_dir.mkdir(parents=True, exist_ok=True)
    downloaded = 0
    skipped = 0
    failed: list[dict] = []
    local_paths: dict[str, str] = {}

    for doc in docs:
        dest = documents_dir / doc.split / doc.file_name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists() and dest.stat().st_size > 0:
            skipped += 1
            local_paths[doc.doc_key] = str(dest)
            continue
        try:
            # Download this PDF
            cached = hf_hub_download(
                repo_id=repo_id,
                repo_type="dataset",
                filename=doc.repo_path,
            )
            dest.write_bytes(Path(cached).read_bytes())


In [ ]:
            downloaded += 1
            local_paths[doc.doc_key] = str(dest)
        except Exception as exc:  # noqa: BLE001 — record and continue
            failed.append({"doc_key": doc.doc_key, "repo_path": doc.repo_path, "error": str(exc)})

    return {
        "requested": len(docs),
        "downloaded": downloaded,
        "skipped_existing": skipped,
        "failed": failed,
        "local_paths": local_paths,
    }


## `extract.py`

**File:** `V2/src/retrieval/extract.py`

**What this file does**  
Reads text out of each downloaded PDF page.

**What goes in**  
A local PDF file path.

**What comes out**  
A list of pages with cleaned text and the page number.

**What I can say in the viva**  
PDF extraction uses PyMuPDF (`fitz`). Empty pages are skipped.


In [ ]:
"""Extract text from FinQA page PDFs."""

from __future__ import annotations

from pathlib import Path
from typing import Any


def clean_text(text: str) -> str:
    # Clean the extracted text
    return " ".join(str(text).replace("\x00", " ").split())


def extract_pdf_pages(pdf_path: Path) -> list[dict[str, Any]]:
    """Return one page dict per PDF page with text + base metadata."""
    try:
        import fitz  # PyMuPDF
    except ImportError as exc:
        raise RuntimeError("Install PyMuPDF: pip install pymupdf") from exc

    pages: list[dict[str, Any]] = []
    # Open the PDF
    document = fitz.open(pdf_path)
    try:
        for page_index, page in enumerate(document, start=1):
            # Read text from this page
            text = clean_text(page.get_text("text"))
            if not text:
                continue
            pages.append(
                {
                    "text": text,
                    "page": page_index,
                    "local_path": str(pdf_path),
                }
            )
    finally:
        document.close()
    return pages


## `chunking.py`

**File:** `V2/src/retrieval/chunking.py`

**What this file does**  
Cuts page text into overlapping windows so retrieval can search smaller pieces.

**What goes in**  
Page text, chunk size 900, overlap 150, plus document metadata.

**What comes out**  
Chunks of text, each still carrying PDF provenance (file, page, company, role).

**What I can say in the viva**  
Chunking is character-based, not token-based. The overlap is there so a number is less likely to be split across two chunks.


In [ ]:
"""Simple character chunking for KB indexing."""

from __future__ import annotations

from typing import Any


# Split into overlapping chunks
def split_text(text: str, chunk_size: int = 900, chunk_overlap: int = 150) -> list[str]:
    """Sliding-window character chunks with overlap."""
    text = " ".join(str(text).split())
    if not text:
        return []
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if chunk_overlap < 0 or chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be >= 0 and < chunk_size")
    if len(text) <= chunk_size:
        return [text]

    chunks: list[str] = []
    start = 0
    # Move window with overlap
    step = chunk_size - chunk_overlap
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start += step
    return chunks


# Chunk every page
def chunk_pages(
    pages: list[dict[str, Any]],
    *,
    chunk_size: int = 900,
    chunk_overlap: int = 150,
    base_metadata: dict[str, Any] | None = None,
) -> list[dict[str, Any]]:
    """Chunk page texts; each chunk carries provenance metadata."""
    base_metadata = base_metadata or {}
    chunks: list[dict[str, Any]] = []
    for page in pages:
        pieces = split_text(page["text"], chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        for idx, piece in enumerate(pieces):
            meta = {
                **base_metadata,
                "page": int(page.get("page") or 1),
                "local_path": str(page.get("local_path") or ""),
                "chunk_index": idx,
                "source_type": "pdf",
            }
            chunks.append({"text": piece, "metadata": meta})
    return chunks


## `embeddings.py`

**File:** `V2/src/retrieval/embeddings.py`

**What this file does**  
Turns chunk text (and later the question) into vectors using BGE-small.

**What goes in**  
A list of text strings and the model name `BAAI/bge-small-en-v1.5`.

**What comes out**  
A list of embedding vectors.

**What I can say in the viva**  
The LLM does not embed the knowledge base. Embeddings are a separate smaller model.


In [ ]:
"""Embedding model loader for the knowledge base."""

from __future__ import annotations

from functools import lru_cache
from typing import Any


@lru_cache(maxsize=2)
def get_embedding_model(model_name: str = "BAAI/bge-small-en-v1.5") -> Any:
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError as exc:
        raise RuntimeError(
            "Install sentence-transformers: pip install sentence-transformers"
        ) from exc
    # Load BGE-small
    return SentenceTransformer(model_name)


def embed_texts(
    texts: list[str],
    *,
    model_name: str = "BAAI/bge-small-en-v1.5",
    batch_size: int = 32,
) -> list[list[float]]:
    model = get_embedding_model(model_name)
    # Create embeddings
    vectors = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=len(texts) > 64,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return [vector.tolist() for vector in vectors]


## `index.py`

**File:** `V2/src/retrieval/index.py`

**What this file does**  
Creates the Chroma collection, adds the embedded chunks, and writes the index manifest.

**What goes in**  
Downloaded PDFs, chunk settings, embedding model, persist folder `knowledge_base/index`.

**What comes out**  
A Chroma collection named `finqa_source_pdfs` (cosine space) and `index_manifest.json`.

**What I can say in the viva**  
This is where ChromaDB is created and filled. The manifest note says gold context is not ingested.


In [ ]:
"""Build and load the persistent Chroma knowledge-base index."""

from __future__ import annotations

from datetime import datetime, timezone
import json
from pathlib import Path
from typing import Any

from src.retrieval.chunking import chunk_pages
from src.retrieval.embeddings import embed_texts
from src.retrieval.extract import extract_pdf_pages
from src.retrieval.pdf_fetch import CorpusDoc


COLLECTION_NAME = "finqa_source_pdfs"


def _chroma_client(persist_dir: Path):
    try:
        import chromadb
    except ImportError as exc:
        raise RuntimeError("Install chromadb: pip install chromadb") from exc
    persist_dir.mkdir(parents=True, exist_ok=True)
    # Open the Chroma store
    return chromadb.PersistentClient(path=str(persist_dir))


def load_collection(persist_dir: Path, collection_name: str = COLLECTION_NAME):
    client = _chroma_client(persist_dir)
    return client.get_or_create_collection(name=collection_name, metadata={"hnsw:space": "cosine"})


def _sanitize_metadata(meta: dict[str, Any]) -> dict[str, Any]:
    clean: dict[str, Any] = {}
    for key, value in meta.items():
        if value is None:
            clean[key] = ""
        elif isinstance(value, (str, int, float, bool)):
            clean[key] = value
        else:
            clean[key] = str(value)
    return clean


def build_knowledge_base(
    docs: list[CorpusDoc],
    local_paths: dict[str, str],
    *,
    persist_dir: Path,
    documents_dir: Path,
    chunk_size: int = 900,
    chunk_overlap: int = 150,
    embedding_model: str = "BAAI/bge-small-en-v1.5",
    collection_name: str = COLLECTION_NAME,
    reset: bool = True,
) -> dict[str, Any]:
    """Extract, chunk, embed, and persist Chroma index from downloaded PDFs."""
    client = _chroma_client(persist_dir)
    if reset:
        try:
            client.delete_collection(collection_name)
        except Exception:
            pass
    collection = client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"},
    )

    all_texts: list[str] = []
    all_ids: list[str] = []
    all_metas: list[dict[str, Any]] = []
    docs_indexed = 0
    docs_empty = 0
    extract_failures: list[dict[str, str]] = []

    for doc in docs:
        local = local_paths.get(doc.doc_key)
        if not local:
            extract_failures.append({"doc_key": doc.doc_key, "error": "missing_local_pdf"})
            continue
        pdf_path = Path(local)
        try:
            # Read the PDF
            pages = extract_pdf_pages(pdf_path)
        except Exception as exc:  # noqa: BLE001
            extract_failures.append({"doc_key": doc.doc_key, "error": str(exc)})
            continue
        if not pages:
            docs_empty += 1
            continue

        base_meta = {
            "doc_id": doc.doc_key,
            "split": doc.split,
            "file_name": doc.file_name,
            "repo_pdf_path": doc.repo_path,
            "role": doc.role,
            "company_symbol": doc.company_symbol,
            "company_name": doc.company_name,
            "report_year": doc.report_year,
            "context_id": doc.context_id,
            "question_id": doc.question_id,
            "source_type": "pdf",
        }
        # Split into overlapping chunks
        chunks = chunk_pages(
            pages,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            base_metadata=base_meta,
        )
        if not chunks:
            docs_empty += 1
            continue

        for i, chunk in enumerate(chunks):
            chunk_id = f"{doc.doc_key}::chunk_{i:04d}"
            meta = _sanitize_metadata({**chunk["metadata"], "chunk_id": chunk_id})
            all_ids.append(chunk_id)
            all_texts.append(chunk["text"])
            all_metas.append(meta)
        docs_indexed += 1

    if not all_texts:
        raise RuntimeError("No chunks produced — cannot build knowledge base")

    # Create embeddings
    embeddings = embed_texts(all_texts, model_name=embedding_model)

    # Add in batches to avoid oversized requests.
    batch_size = 100
    for start in range(0, len(all_ids), batch_size):
        end = start + batch_size
        # Store chunks in Chroma
        collection.add(
            ids=all_ids[start:end],
            documents=all_texts[start:end],
            embeddings=embeddings[start:end],
            metadatas=all_metas[start:end],
        )

    manifest = {
        "phase": 6,
        "built_at_utc": datetime.now(timezone.utc).isoformat(),


In [ ]:
        "collection_name": collection_name,
        "persist_dir": str(persist_dir),
        "documents_dir": str(documents_dir),
        "embedding_model": embedding_model,
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "vector_store": "chroma",
        "docs_requested": len(docs),
        "docs_indexed": docs_indexed,
        "docs_empty_text": docs_empty,
        "chunks": len(all_ids),
        "roles": {
            "test": sum(1 for d in docs if d.role == "test"),
            "calibration": sum(1 for d in docs if d.role == "calibration"),
            "distractor": sum(1 for d in docs if d.role == "distractor"),
        },
        "extract_failures": extract_failures,
        "note": (
            "Index is built from FinQA source page PDFs only. "
            "Gold context fields are not ingested as retrieval documents."
        ),
    }
    manifest_path = persist_dir / "index_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    return manifest


## `retriever.py`

**File:** `V2/src/retrieval/retriever.py`

**What this file does**  
At question time: embed the question, query Chroma, return the top chunks.

**What goes in**  
The question, the index folder, `top_k=4`, the same embedding model.

**What comes out**  
A list of `RetrievedChunk` objects with text, score, and metadata.

**What I can say in the viva**  
All three RAG architectures call this same `retrieve()` function. Similarity is `1 - distance`.


In [ ]:
"""Query the persistent knowledge-base index."""

from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

from src.retrieval.embeddings import embed_texts
from src.retrieval.index import COLLECTION_NAME, load_collection


@dataclass
class RetrievedChunk:
    chunk_id: str
    text: str
    score: float
    doc_id: str
    file_name: str
    split: str
    page: int | str
    company_symbol: str
    report_year: str
    role: str
    context_id: str
    source_type: str

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


def _distance_to_similarity(distance: float | None) -> float:
    if distance is None:
        return 0.0
    # Chroma cosine space returns distance; convert to a bounded similarity-like score.
    return max(0.0, 1.0 - float(distance))


def retrieve(
    question: str,
    *,
    persist_dir: Path,
    top_k: int = 4,
    embedding_model: str = "BAAI/bge-small-en-v1.5",
    collection_name: str = COLLECTION_NAME,
) -> list[RetrievedChunk]:
    if not question or not question.strip():
        raise ValueError("question must be non-empty")

    collection = load_collection(persist_dir, collection_name=collection_name)
    # Embed the question
    query_vec = embed_texts([question], model_name=embedding_model)[0]
    # Search for the top 4 chunks
    result = collection.query(
        query_embeddings=[query_vec],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    documents = (result.get("documents") or [[]])[0]
    metadatas = (result.get("metadatas") or [[]])[0]
    distances = (result.get("distances") or [[]])[0]
    ids = (result.get("ids") or [[]])[0]

    chunks: list[RetrievedChunk] = []
    for i, text in enumerate(documents):
        meta = metadatas[i] if i < len(metadatas) else {}
        distance = distances[i] if i < len(distances) else None
        chunk_id = ids[i] if i < len(ids) else str(meta.get("chunk_id") or f"idx_{i}")
        chunks.append(
            RetrievedChunk(
                chunk_id=str(chunk_id),
                text=str(text or ""),
                score=_distance_to_similarity(distance),
                doc_id=str(meta.get("doc_id") or ""),
                file_name=str(meta.get("file_name") or ""),
                split=str(meta.get("split") or ""),
                page=meta.get("page", ""),
                company_symbol=str(meta.get("company_symbol") or ""),
                report_year=str(meta.get("report_year") or ""),
                role=str(meta.get("role") or ""),
                context_id=str(meta.get("context_id") or ""),
                source_type=str(meta.get("source_type") or ""),
            )
        )
    return chunks


# 2. RAG Backend


## `single_agent.py`

**File:** `V2/src/rag/single_agent.py`

**What this file does**  
Baseline RAG: retrieve evidence, build a prompt, generate an answer with Qwen. Always answers.

**What goes in**  
A question, the shared Chroma index, the Qwen backend.

**What comes out**  
One `RAGCaseResult`: retrieved evidence, generated answer, `decision='ANSWER'`, no verification, no confidence.

**What I can say in the viva**  
This is architecture 1. There is no checker and no abstention gate.


In [ ]:
"""Single-Agent RAG baseline (Phase 8).

Retrieve from the Phase 6 source-PDF knowledge base, then generate with the
configured Qwen3-8B backend. No multi-agent verification or abstention.
"""

from __future__ import annotations

import time
from pathlib import Path
from typing import Any

from src.config import ExperimentConfig, get_path, load_experiment_config, load_prompts_config
from src.models.factory import create_backend
from src.models.fingerprint import collect_fingerprint
from src.models.types import LLMBackend
from src.rag.prompts import build_baseline_prompt
from src.rag.schema import ARCHITECTURE_SINGLE_AGENT, RAGCaseResult
from src.rag.text_utils import clean_generated_answer
from src.retrieval.retriever import retrieve
from src.utils import create_run_id


def run_single_agent(
    question: str,
    *,
    question_id: str = "adhoc",
    reference_answer: str | None = None,
    config: ExperimentConfig | None = None,
    backend: LLMBackend | None = None,
    backend_name: str | None = None,
    run_id: str | None = None,
    fingerprint: dict[str, Any] | None = None,
) -> RAGCaseResult:
    """Run retrieve → prompt → generate for one question."""
    cfg = config or load_experiment_config()
    model_cfg = dict(cfg.section("model"))
    retrieval_cfg = cfg.section("retrieval")
    embeddings_cfg = cfg.section("embeddings")
    execution_cfg = cfg.section("execution")

    if backend_name:
        model_cfg["backend"] = backend_name

    from src.config import project_root

    rid = run_id or create_run_id("phase8")
    architecture = ARCHITECTURE_SINGLE_AGENT
    case_key = f"{architecture}:{question_id}"

    top_k = int(retrieval_cfg.get("top_k") or 4)
    persist_dir = get_path(cfg, "kb_index")
    embed_model = str(embeddings_cfg.get("model") or "BAAI/bge-small-en-v1.5")
    collection = str(retrieval_cfg.get("collection_name") or "finqa_source_pdfs")

    fp = fingerprint or collect_fingerprint(
        model_config=model_cfg,
        project_root=str(project_root()),
    )

    llm = backend or create_backend(model_cfg)
    start = time.perf_counter()
    error: str | None = None
    answer = ""
    chunks_dicts: list[dict[str, Any]] = []
    scores: list[float] = []
    prompt_chars = 0

    try:
        # Search for the top 4 chunks
        chunks = retrieve(
            question,
            persist_dir=persist_dir,
            top_k=top_k,
            embedding_model=embed_model,
            collection_name=collection,
        )
        chunks_dicts = [c.to_dict() for c in chunks]
        scores = [float(c.score) for c in chunks]
        prompts_cfg = load_prompts_config()
        # Build the prompt
        prompt = build_baseline_prompt(question, chunks, prompts_cfg=prompts_cfg)
        prompt_chars = len(prompt)
        # Generate the answer
        gen = llm.generate(
            prompt,
            temperature=float(model_cfg.get("temperature") or 0.1),
            max_new_tokens=int(model_cfg.get("max_new_tokens") or 512),
            top_p=model_cfg.get("top_p"),
        )
        answer = clean_generated_answer(gen.text or "")
        # One retry if the backend returns empty text (seen with some local Ollama/Qwen3 runs).
        if not answer:
            gen = llm.generate(
                prompt,
                temperature=float(model_cfg.get("temperature") or 0.1),
                max_new_tokens=int(model_cfg.get("max_new_tokens") or 512),
                top_p=model_cfg.get("top_p"),
            )
            answer = clean_generated_answer(gen.text or "")
        model_name = gen.model
        quant = gen.quantisation
        backend_used = gen.backend
        if not answer:
            error = "Generation returned empty text after retry"
    except Exception as exc:  # noqa: BLE001 — capture into result record
        error = f"{type(exc).__name__}: {exc}"
        model_name = str(model_cfg.get("name") or "Qwen3-8B")
        quant = model_cfg.get("quantisation")
        backend_used = getattr(llm, "name", model_cfg.get("backend"))

    latency = time.perf_counter() - start
    gpu_info = (fp or {}).get("gpu") or {}
    gpu_name = gpu_info.get("name") if gpu_info.get("available") else None

    return RAGCaseResult(
        run_id=rid,
        question_id=str(question_id),
        architecture=architecture,
        question=question,
        retrieved_evidence=chunks_dicts,
        retrieval_scores=scores,
        answer=answer,
        reference_answer=reference_answer,
        # No verify or abstain here
        verification_result=None,
        confidence=None,
        threshold=None,
        decision="ANSWER",
        latency_seconds=latency,
        model=model_name,
        model_version=None,
        quantisation=str(quant) if quant is not None else None,
        device=(fp or {}).get("device"),
        gpu=gpu_name,
        configuration={
            "top_k": top_k,
            "embedding_model": embed_model,
            "collection_name": collection,
            "persist_dir": str(persist_dir),
            "hf_repo_id": model_cfg.get("hf_repo_id"),
            "gguf_filename": model_cfg.get("gguf_filename"),
            "temperature": model_cfg.get("temperature"),
            "max_new_tokens": model_cfg.get("max_new_tokens"),


In [ ]:
        },
        random_seed=execution_cfg.get("random_seed"),
        error=error,
        retrieval_top_k=top_k,
        backend=str(backend_used) if backend_used else None,
        prompt_chars=prompt_chars,
        case_key=case_key,
    )


## `multi_agent.py`

**File:** `V2/src/rag/multi_agent.py`

**What this file does**  
Retrieve, generate a draft, then verify the draft against the evidence. Still always answers.

**What goes in**  
The same question, same index, same Qwen backend as Single-Agent.

**What comes out**  
A `RAGCaseResult` with the draft as `answer` plus a `verification_result`. Decision stays `ANSWER`.

**What I can say in the viva**  
This is architecture 2. The checker does not rewrite the answer. There is no LangGraph. The flow is ordinary Python.


In [ ]:
"""Multi-Agent RAG (Phase 9): retrieve → draft → verify.

Uses the shared Phase 6 knowledge base and Qwen3-8B backend.
No uncertainty quantification or abstention (Phase 10).
"""

from __future__ import annotations

import time
from typing import Any

from src.config import ExperimentConfig, get_path, load_experiment_config, load_prompts_config
from src.models.factory import create_backend
from src.models.fingerprint import collect_fingerprint
from src.models.types import GenerationResult, LLMBackend
from src.rag.prompts import build_multi_agent_draft_prompt, build_multi_agent_verification_prompt
from src.rag.schema import ARCHITECTURE_MULTI_AGENT, RAGCaseResult
from src.rag.text_utils import clean_generated_answer
from src.rag.verification import compute_verification_result
from src.retrieval.retriever import retrieve
from src.utils import create_run_id


def _generate_with_retry(
    llm: LLMBackend,
    prompt: str,
    *,
    temperature: float,
    max_new_tokens: int,
    top_p: float | None,
) -> GenerationResult:
    gen = llm.generate(
        prompt,
        temperature=temperature,
        max_new_tokens=max_new_tokens,
        top_p=top_p,
    )
    if not (gen.text or "").strip():
        gen = llm.generate(
            prompt,
            temperature=temperature,
            max_new_tokens=max_new_tokens,
            top_p=top_p,
        )
    return gen


def run_multi_agent(
    question: str,
    *,
    question_id: str = "adhoc",
    reference_answer: str | None = None,
    config: ExperimentConfig | None = None,
    backend: LLMBackend | None = None,
    backend_name: str | None = None,
    run_id: str | None = None,
    fingerprint: dict[str, Any] | None = None,
) -> RAGCaseResult:
    """Run retrieve → draft → verification for one question."""
    cfg = config or load_experiment_config()
    model_cfg = dict(cfg.section("model"))
    retrieval_cfg = cfg.section("retrieval")
    embeddings_cfg = cfg.section("embeddings")
    execution_cfg = cfg.section("execution")
    rag_cfg = cfg.section("rag")
    multi_cfg = dict(rag_cfg.get("architectures", {}).get("multi_agent") or {})

    if backend_name:
        model_cfg["backend"] = backend_name

    from src.config import project_root

    rid = run_id or create_run_id("phase9")
    architecture = ARCHITECTURE_MULTI_AGENT
    case_key = f"{architecture}:{question_id}"

    top_k = int(retrieval_cfg.get("top_k") or 4)
    persist_dir = get_path(cfg, "kb_index")
    embed_model = str(embeddings_cfg.get("model") or "BAAI/bge-small-en-v1.5")
    collection = str(retrieval_cfg.get("collection_name") or "finqa_source_pdfs")
    verify_threshold = float(multi_cfg.get("verification_threshold") or 0.5)
    temperature = float(model_cfg.get("temperature") or 0.1)
    max_new_tokens = int(model_cfg.get("max_new_tokens") or 512)
    top_p = model_cfg.get("top_p")

    fp = fingerprint or collect_fingerprint(
        model_config=model_cfg,
        project_root=str(project_root()),
    )

    llm = backend or create_backend(model_cfg)
    start = time.perf_counter()
    error: str | None = None
    answer = ""
    chunks_dicts: list[dict[str, Any]] = []
    scores: list[float] = []
    draft_prompt_chars = 0
    verify_prompt_chars = 0
    verification_result: dict[str, Any] | None = None
    model_name = str(model_cfg.get("name") or "Qwen3-8B")
    quant: Any = model_cfg.get("quantisation")
    backend_used: Any = getattr(llm, "name", model_cfg.get("backend"))

    try:
        # Search for the top 4 chunks
        chunks = retrieve(
            question,
            persist_dir=persist_dir,
            top_k=top_k,
            embedding_model=embed_model,
            collection_name=collection,
        )
        chunks_dicts = [c.to_dict() for c in chunks]
        scores = [float(c.score) for c in chunks]

        prompts_cfg = load_prompts_config()
        # Build the draft prompt
        draft_prompt = build_multi_agent_draft_prompt(question, chunks, prompts_cfg=prompts_cfg)
        draft_prompt_chars = len(draft_prompt)
        # Generate the draft answer
        draft_gen = _generate_with_retry(
            llm,
            draft_prompt,
            temperature=temperature,
            max_new_tokens=max_new_tokens,
            top_p=top_p,
        )
        answer = clean_generated_answer(draft_gen.text or "")
        model_name = draft_gen.model
        quant = draft_gen.quantisation
        backend_used = draft_gen.backend

        if not answer:
            error = "Draft generation returned empty text after retry"
        else:
            # Build the verify prompt
            verify_prompt = build_multi_agent_verification_prompt(
                question,
                answer,
                chunks,
                prompts_cfg=prompts_cfg,
            )
            verify_prompt_chars = len(verify_prompt)
            # Check whether the answer is supported
            verification_result = compute_verification_result(


In [ ]:
                question,
                answer,
                chunks,
                llm,
                prompts_cfg=prompts_cfg,
                verification_threshold=verify_threshold,
            )
    except Exception as exc:  # noqa: BLE001
        error = f"{type(exc).__name__}: {exc}"

    latency = time.perf_counter() - start
    gpu_info = (fp or {}).get("gpu") or {}
    gpu_name = gpu_info.get("name") if gpu_info.get("available") else None
    confidence = (
        float(verification_result["verification_score"])
        if verification_result is not None
        else None
    )

    return RAGCaseResult(
        run_id=rid,
        question_id=str(question_id),
        architecture=architecture,
        question=question,
        retrieved_evidence=chunks_dicts,
        retrieval_scores=scores,
        answer=answer,
        reference_answer=reference_answer,
        verification_result=verification_result,
        # Always ANSWER (no abstain here)
        confidence=confidence,
        threshold=None,
        decision="ANSWER",
        latency_seconds=latency,
        model=model_name,
        model_version=None,
        quantisation=str(quant) if quant is not None else None,
        device=(fp or {}).get("device"),
        gpu=gpu_name,
        configuration={
            "top_k": top_k,
            "embedding_model": embed_model,
            "collection_name": collection,
            "persist_dir": str(persist_dir),
            "hf_repo_id": model_cfg.get("hf_repo_id"),
            "gguf_filename": model_cfg.get("gguf_filename"),
            "temperature": temperature,
            "max_new_tokens": max_new_tokens,
            "verification_threshold": verify_threshold,
            "draft_prompt_chars": draft_prompt_chars,
            "verify_prompt_chars": verify_prompt_chars,
        },
        random_seed=execution_cfg.get("random_seed"),
        error=error,
        retrieval_top_k=top_k,
        backend=str(backend_used) if backend_used else None,
        prompt_chars=draft_prompt_chars,
        case_key=case_key,
    )


## `verification.py`

**File:** `V2/src/rag/verification.py`

**What this file does**  
Scores whether the draft is supported by the retrieved evidence.

**What goes in**  
The question, the draft answer, the retrieved chunks, and the LLM.

**What comes out**  
A dictionary: lexical score, LLM support score, combined `verification_score`, status `VERIFIED` or `WEAK_EVIDENCE`, and a rationale.

**What I can say in the viva**  
Verification is the mean of token overlap and an LLM 0–1 support score. Status uses threshold 0.50. This file does not abstain.


In [ ]:
"""Evidence-grounded verification for Multi-Agent RAG (Phase 9)."""

from __future__ import annotations

from typing import Any

from src.models.types import LLMBackend
from src.rag.prompts import build_multi_agent_verification_prompt, format_evidence
from src.rag.text_utils import average, build_verification_rationale, parse_unit_score, token_overlap
from src.retrieval.retriever import RetrievedChunk


def compute_verification_result(
    question: str,
    answer: str,
    chunks: list[RetrievedChunk],
    llm: LLMBackend,
    *,
    prompts_cfg: dict[str, Any] | None = None,
    verification_threshold: float = 0.5,
    temperature: float = 0.0,
    max_new_tokens: int = 32,
) -> dict[str, Any]:
    """Lexical overlap + LLM support score (no abstention logic here)."""
    evidence_text = format_evidence(chunks)
    # How much the answer overlaps evidence
    lexical_score = token_overlap(answer, evidence_text)
    llm_score: float | None = None

    if answer.strip():
        # Ask if the evidence supports it
        prompt = build_multi_agent_verification_prompt(
            question,
            answer,
            chunks,
            prompts_cfg=prompts_cfg,
        )
        gen = llm.generate(
            prompt,
            temperature=temperature,
            max_new_tokens=max_new_tokens,
        )
        # Parse the 0 to 1 support score
        llm_score = parse_unit_score(gen.text or "")

    if llm_score is not None:
        # Average the two support scores
        verification_score = average([lexical_score, llm_score])
    else:
        verification_score = lexical_score

    # Mark verified or weak
    status = "VERIFIED" if verification_score >= verification_threshold else "WEAK_EVIDENCE"
    rationale = build_verification_rationale(
        status=status,
        verification_score=verification_score,
        lexical_score=lexical_score,
        llm_score=llm_score,
        verification_threshold=verification_threshold,
    )
    return {
        "verification_score": verification_score,
        "lexical_score": lexical_score,
        "llm_score": llm_score,
        "verification_threshold": verification_threshold,
        "status": status,
        "rationale": rationale,
    }


## `uncertainty.py`

**File:** `V2/src/rag/uncertainty.py`

**What this file does**  
Turns retrieval and verification into one confidence number, then chooses ANSWER or ABSTAIN.

**What goes in**  
The retrieval scores, the verification score, the draft answer, and the threshold T.

**What comes out**  
A confidence dictionary, then the final answer text and a decision (`ANSWER` or `ABSTAIN`).

**What I can say in the viva**  
Confidence is the mean of retrieval score and verification score. It is a decision score, not a calibrated probability. If confidence is below T, the system abstains.


In [ ]:
"""Uncertainty quantification and abstention for Multi-Agent + UQ (Phase 10)."""

from __future__ import annotations

from typing import Any

from src.rag.text_utils import average


def compute_retrieval_score(retrieval_scores: list[float]) -> float:
    """Aggregate top-k retrieval similarities into one signal (mean)."""
    if not retrieval_scores:
        return 0.0
    # Average the retrieval scores
    return average(retrieval_scores)


def compute_combined_confidence(
    retrieval_score: float,
    verification_score: float,
    *,
    method: str = "mean_retrieval_verification",
) -> dict[str, Any]:
    """Combine retrieval and verification into a single confidence score."""
    if method != "mean_retrieval_verification":
        raise ValueError(f"Unsupported UQ method: {method}")
    # Calculate confidence
    confidence = average([retrieval_score, verification_score])
    return {
        "method": method,
        "retrieval_score": retrieval_score,
        "verification_score": verification_score,
        "confidence": confidence,
    }


def apply_abstention_decision(
    *,
    draft_answer: str,
    confidence: float,
    threshold: float,
    abstention_message: str,
) -> tuple[str, str]:
    """Return (final_answer, decision) where decision is ANSWER or ABSTAIN."""
    # Apply the threshold
    if confidence >= threshold:
        # Return ANSWER
        return draft_answer, "ANSWER"
    # Return ABSTAIN
    return abstention_message, "ABSTAIN"


## `multi_agent_uq.py`

**File:** `V2/src/rag/multi_agent_uq.py`

**What this file does**  
Architecture 3: retrieve, draft, verify, compute confidence, then apply the abstention gate.

**What goes in**  
The question, the shared index, Qwen, and a threshold (locked T = 0.65 in the official run).

**What comes out**  
A `RAGCaseResult` whose displayed `answer` may be the draft or the abstention message. The draft is kept in `configuration['draft_answer']`.

**What I can say in the viva**  
This is Multi-Agent + UQ. It reuses retrieval and verification, then adds the confidence gate. The three architectures are independent: they do not pass answers to each other.


In [ ]:
"""Multi-Agent RAG + UQ / abstention (Phase 10): retrieve → draft → verify → confidence gate.

Extends Phase 9 with combined confidence (retrieval + verification) and ANSWER | ABSTAIN.
No self-consistency sampling (avoid V1 cost/constant-confidence issue).
"""

from __future__ import annotations

import time
from typing import Any

from src.config import ExperimentConfig, get_path, load_experiment_config, load_prompts_config
from src.models.factory import create_backend
from src.models.fingerprint import collect_fingerprint
from src.models.types import LLMBackend
from src.rag.multi_agent import _generate_with_retry
from src.rag.prompts import build_multi_agent_draft_prompt, build_multi_agent_verification_prompt
from src.rag.schema import ARCHITECTURE_MULTI_AGENT_UQ, RAGCaseResult
from src.rag.text_utils import clean_generated_answer
from src.rag.uncertainty import (
    apply_abstention_decision,
    compute_combined_confidence,
    compute_retrieval_score,
)
from src.rag.verification import compute_verification_result
from src.retrieval.retriever import retrieve
from src.utils import create_run_id


# Read T from the lock
def _resolve_threshold(cfg: ExperimentConfig, threshold_override: float | None) -> float:
    if threshold_override is not None:
        return float(threshold_override)
    uq_cfg = cfg.section("uncertainty")
    locked = uq_cfg.get("confidence_threshold")
    if locked is not None:
        return float(locked)
    smoke = uq_cfg.get("smoke_threshold")
    if smoke is not None:
        return float(smoke)
    return 0.55


def run_multi_agent_uq(
    question: str,
    *,
    question_id: str = "adhoc",
    reference_answer: str | None = None,
    config: ExperimentConfig | None = None,
    backend: LLMBackend | None = None,
    backend_name: str | None = None,
    run_id: str | None = None,
    fingerprint: dict[str, Any] | None = None,
    threshold: float | None = None,
) -> RAGCaseResult:
    """Run retrieve → draft → verify → combined confidence → abstention gate."""
    cfg = config or load_experiment_config()
    model_cfg = dict(cfg.section("model"))
    retrieval_cfg = cfg.section("retrieval")
    embeddings_cfg = cfg.section("embeddings")
    execution_cfg = cfg.section("execution")
    rag_cfg = cfg.section("rag")
    uq_cfg = cfg.section("uncertainty")
    multi_cfg = dict(rag_cfg.get("architectures", {}).get("multi_agent") or {})
    prompts_cfg = load_prompts_config()
    uq_prompts = dict(prompts_cfg.get("uncertainty") or {})

    if backend_name:
        model_cfg["backend"] = backend_name

    from src.config import project_root

    rid = run_id or create_run_id("phase10")
    architecture = ARCHITECTURE_MULTI_AGENT_UQ
    case_key = f"{architecture}:{question_id}"

    top_k = int(retrieval_cfg.get("top_k") or 4)
    persist_dir = get_path(cfg, "kb_index")
    embed_model = str(embeddings_cfg.get("model") or "BAAI/bge-small-en-v1.5")
    collection = str(retrieval_cfg.get("collection_name") or "finqa_source_pdfs")
    verify_threshold = float(multi_cfg.get("verification_threshold") or 0.5)
    # This is T = 0.65 in the viva
    confidence_threshold = _resolve_threshold(cfg, threshold)
    uq_method = str(uq_cfg.get("method") or "mean_retrieval_verification")
    abstention_message = str(
        uq_prompts.get("abstention_message")
        or uq_cfg.get("abstention_message")
        or "I cannot answer reliably because supporting evidence is insufficient."
    )
    temperature = float(model_cfg.get("temperature") or 0.1)
    max_new_tokens = int(model_cfg.get("max_new_tokens") or 512)
    top_p = model_cfg.get("top_p")

    fp = fingerprint or collect_fingerprint(
        model_config=model_cfg,
        project_root=str(project_root()),
    )

    llm = backend or create_backend(model_cfg)
    start = time.perf_counter()
    error: str | None = None
    draft_answer = ""
    final_answer = ""
    chunks_dicts: list[dict[str, Any]] = []
    scores: list[float] = []
    draft_prompt_chars = 0
    verify_prompt_chars = 0
    verification_result: dict[str, Any] | None = None
    uncertainty_result: dict[str, Any] | None = None
    decision = "ANSWER"
    confidence: float | None = None
    model_name = str(model_cfg.get("name") or "Qwen3-8B")
    quant: Any = model_cfg.get("quantisation")
    backend_used: Any = getattr(llm, "name", model_cfg.get("backend"))

    try:
        # Search for the top 4 chunks
        chunks = retrieve(
            question,
            persist_dir=persist_dir,
            top_k=top_k,
            embedding_model=embed_model,
            collection_name=collection,
        )
        chunks_dicts = [c.to_dict() for c in chunks]
        scores = [float(c.score) for c in chunks]

        # Build the draft prompt
        draft_prompt = build_multi_agent_draft_prompt(question, chunks, prompts_cfg=prompts_cfg)
        draft_prompt_chars = len(draft_prompt)
        # Generate the draft answer
        draft_gen = _generate_with_retry(
            llm,
            draft_prompt,
            temperature=temperature,
            max_new_tokens=max_new_tokens,
            top_p=top_p,
        )
        draft_answer = clean_generated_answer(draft_gen.text or "")
        model_name = draft_gen.model
        quant = draft_gen.quantisation
        backend_used = draft_gen.backend

        if not draft_answer:
            error = "Draft generation returned empty text after retry"


In [ ]:
        else:
            # Build the verify prompt
            verify_prompt = build_multi_agent_verification_prompt(
                question,
                draft_answer,
                chunks,
                prompts_cfg=prompts_cfg,
            )
            verify_prompt_chars = len(verify_prompt)
            # Check whether the answer is supported
            verification_result = compute_verification_result(
                question,
                draft_answer,
                chunks,
                llm,
                prompts_cfg=prompts_cfg,
                verification_threshold=verify_threshold,
            )
            # Average retrieval scores
            retrieval_score = compute_retrieval_score(scores)
            verify_score = float(verification_result["verification_score"])
            # Calculate confidence
            uncertainty_result = compute_combined_confidence(
                retrieval_score,
                verify_score,
                method=uq_method,
            )
            confidence = float(uncertainty_result["confidence"])
            # Apply the 0.65 threshold
            final_answer, decision = apply_abstention_decision(
                draft_answer=draft_answer,
                confidence=confidence,
                threshold=confidence_threshold,
                abstention_message=abstention_message,
            )
    except Exception as exc:  # noqa: BLE001
        error = f"{type(exc).__name__}: {exc}"

    latency = time.perf_counter() - start
    gpu_info = (fp or {}).get("gpu") or {}
    gpu_name = gpu_info.get("name") if gpu_info.get("available") else None

    return RAGCaseResult(
        run_id=rid,
        question_id=str(question_id),
        architecture=architecture,
        question=question,
        retrieved_evidence=chunks_dicts,
        retrieval_scores=scores,
        answer=final_answer if not error else "",
        reference_answer=reference_answer,
        verification_result=verification_result,
        confidence=confidence,
        threshold=confidence_threshold,
        decision=decision,
        latency_seconds=latency,
        model=model_name,
        model_version=None,
        quantisation=str(quant) if quant is not None else None,
        device=(fp or {}).get("device"),
        gpu=gpu_name,
        configuration={
            "top_k": top_k,
            "embedding_model": embed_model,
            "collection_name": collection,
            "persist_dir": str(persist_dir),
            "hf_repo_id": model_cfg.get("hf_repo_id"),
            "gguf_filename": model_cfg.get("gguf_filename"),
            "temperature": temperature,
            "max_new_tokens": max_new_tokens,
            "verification_threshold": verify_threshold,
            "draft_prompt_chars": draft_prompt_chars,
            "verify_prompt_chars": verify_prompt_chars,
            "draft_answer": draft_answer or None,
            "uncertainty_result": uncertainty_result,
            "uq_method": uq_method,
            "threshold_source": (
                "override"
                if threshold is not None
                else ("locked" if uq_cfg.get("confidence_threshold") is not None else "smoke")
            ),
        },
        random_seed=execution_cfg.get("random_seed"),
        error=error,
        retrieval_top_k=top_k,
        backend=str(backend_used) if backend_used else None,
        prompt_chars=draft_prompt_chars,
        case_key=case_key,
    )


# 3. Prompt Backend


## `prompts.py`

**File:** `V2/src/rag/prompts.py`

**What this file does**  
Builds the actual prompt strings for baseline generation, Multi-Agent draft, and verification.

**What goes in**  
The question, retrieved chunks, the draft answer for verification, and `config/prompts.yaml`.

**What comes out**  
One string prompt for Qwen.

**What I can say in the viva**  
The prompts tell the model to use only the evidence, write the answer once, and distinguish final value vs change vs ROI. If evidence is missing, the instructed fallback is `Evidence is insufficient.`


In [ ]:
"""Prompt construction for V2 RAG architectures."""

from __future__ import annotations

from typing import Any

from src.config import load_prompts_config
from src.retrieval.retriever import RetrievedChunk


DEFAULT_BASELINE_SYSTEM = (
    "Answer the financial question using only the evidence below. "
    "Write the final answer once, in one short sentence or one number with its unit. "
    "Do not repeat the answer. Do not repeat these instructions. Do not write reasoning. "
    "Distinguish the quantity the question asks for: final/ending/cumulative value is not "
    "the same as absolute change, and neither is the same as percentage change or ROI. "
    "If the question asks for ROI or percentage change, do not report the ending investment value. "
    "If the evidence does not contain the answer, write exactly: Evidence is insufficient."
)

DEFAULT_BASELINE_USER = """Evidence:
{evidence}

Question:
{question}

Final answer (once only):"""

DEFAULT_MULTI_AGENT_DRAFT_SYSTEM = (
    "Draft a financial answer using only the evidence below. "
    "Write the final answer once, in one short sentence or one number with its unit. "
    "Do not repeat the answer. Do not repeat these instructions. Do not write reasoning. "
    "Distinguish the quantity the question asks for: final/ending/cumulative value is not "
    "the same as absolute change, and neither is the same as percentage change or ROI. "
    "If the question asks for ROI or percentage change, do not report the ending investment value. "
    "If the evidence does not contain the answer, write exactly: Evidence is insufficient."
)

DEFAULT_MULTI_AGENT_DRAFT_USER = """Evidence:
{evidence}

Question:
{question}

Final answer (once only):"""

DEFAULT_MULTI_AGENT_VERIFY_SYSTEM = (
    "Score how well the draft answer is supported by the evidence and "
    "whether it reports the quantity the question asked for "
    "(final value vs absolute change vs percentage change/ROI). "
    "Reply with only one number from 0.00 to 1.00. "
    "Do not repeat these instructions and do not write words before or after the number."
)

DEFAULT_MULTI_AGENT_VERIFY_USER = """Evidence:
{evidence}

Question:
{question}

Draft answer:
{answer}

Support score:"""


# Put retrieved chunks into the prompt
def format_evidence(chunks: list[RetrievedChunk] | list[dict[str, Any]]) -> str:
    parts: list[str] = []
    for i, chunk in enumerate(chunks, start=1):
        if isinstance(chunk, dict):
            text = str(chunk.get("text") or "")
            file_name = str(chunk.get("file_name") or "")
            score = chunk.get("score")
        else:
            text = chunk.text
            file_name = chunk.file_name
            score = chunk.score
        header = f"[{i}] file={file_name} score={score:.4f}" if score is not None else f"[{i}] file={file_name}"
        parts.append(f"{header}\n{text.strip()}")
    return "\n\n".join(parts) if parts else "(no evidence retrieved)"


# Build the Single-Agent prompt
def build_baseline_prompt(
    question: str,
    chunks: list[RetrievedChunk] | list[dict[str, Any]],
    *,
    prompts_cfg: dict[str, Any] | None = None,
) -> str:
    cfg = prompts_cfg if prompts_cfg is not None else load_prompts_config()
    baseline = cfg.get("baseline") or {}
    system = baseline.get("system") or DEFAULT_BASELINE_SYSTEM
    user_template = baseline.get("user_template") or DEFAULT_BASELINE_USER
    user = user_template.format(
        evidence=format_evidence(chunks),
        question=question.strip(),
    )
    return f"{system.strip()}\n\n{user.strip()}"


# Build the Multi-Agent draft prompt
def build_multi_agent_draft_prompt(
    question: str,
    chunks: list[RetrievedChunk] | list[dict[str, Any]],
    *,
    prompts_cfg: dict[str, Any] | None = None,
) -> str:
    cfg = prompts_cfg if prompts_cfg is not None else load_prompts_config()
    section = (cfg.get("multi_agent") or {}).get("generation") or {}
    system = section.get("system") or DEFAULT_MULTI_AGENT_DRAFT_SYSTEM
    user_template = section.get("user_template") or DEFAULT_MULTI_AGENT_DRAFT_USER
    user = user_template.format(
        evidence=format_evidence(chunks),
        question=question.strip(),
    )
    return f"{system.strip()}\n\n{user.strip()}"


# Build the verify prompt
def build_multi_agent_verification_prompt(
    question: str,
    answer: str,
    chunks: list[RetrievedChunk] | list[dict[str, Any]],
    *,
    prompts_cfg: dict[str, Any] | None = None,
) -> str:
    cfg = prompts_cfg if prompts_cfg is not None else load_prompts_config()
    section = (cfg.get("multi_agent") or {}).get("verification") or {}
    system = section.get("system") or DEFAULT_MULTI_AGENT_VERIFY_SYSTEM
    user_template = section.get("user_template") or DEFAULT_MULTI_AGENT_VERIFY_USER
    user = user_template.format(
        evidence=format_evidence(chunks),
        question=question.strip(),
        answer=answer.strip(),
    )
    return f"{system.strip()}\n\n{user.strip()}"


## `prompts.yaml`

**File:** `V2/config/prompts.yaml`

**What this file does**  
The stored prompt templates used by `prompts.py`, including the judge template.

**What goes in**  
Loaded by `load_prompts_config()`.

**What comes out**  
YAML sections: `baseline`, `multi_agent.generation`, `multi_agent.verification`, `judge`, `uncertainty`.

**What I can say in the viva**  
If asked where the wording lives, this is the file. The judge label in this file is custom / RAGAS-inspired, not official RAGAS.


```yaml
version: "0.3.1-phase11-output"

# Single-Agent prompt
baseline:
  system: >-
    Answer the financial question using only the evidence below.
    Write the final answer once, in one short sentence or one number with its unit.
    Do not repeat the answer. Do not repeat these instructions. Do not write reasoning.
    Distinguish the quantity the question asks for:
    final/ending/cumulative value is not the same as absolute change,
    and neither is the same as percentage change or ROI.
    If the question asks for ROI or percentage change, do not report the ending investment value.
    If the evidence does not contain the answer, write exactly:
    Evidence is insufficient.
  user_template: |
    Evidence:
    {evidence}

    Question:
    {question}

    Final answer (once only):

multi_agent:
  # Multi-Agent draft prompt
  generation:
    system: >-
      Draft a financial answer using only the evidence below.
      Write the final answer once, in one short sentence or one number with its unit.
      Do not repeat the answer. Do not repeat these instructions. Do not write reasoning.
      Distinguish the quantity the question asks for:
      final/ending/cumulative value is not the same as absolute change,
      and neither is the same as percentage change or ROI.
      If the question asks for ROI or percentage change, do not report the ending investment value.
      If the evidence does not contain the answer, write exactly:
      Evidence is insufficient.
    user_template: |
      Evidence:
      {evidence}

      Question:
      {question}

      Final answer (once only):
  # Check if the answer is supported
  verification:
    system: >-
      Score how well the draft answer is supported by the evidence and
      whether it reports the quantity the question asked for
      (final value vs absolute change vs percentage change/ROI).
      Reply with only one number from 0.00 to 1.00.
      Do not repeat these instructions and do not write words before or after the number.
    user_template: |
      Evidence:
      {evidence}

      Question:
      {question}

      Draft answer:
      {answer}

      Support score:

# Judge prompt (not official RAGAS)
judge:
  notes: "Phase 16 post-hoc faithfulness judge. Not official RAGAS. Does not use gold context or gold answers."
  metric_label: "LLM-as-judge faithfulness (Qwen3-8B, custom/RAGAS-inspired)"
  system: >-
    You are scoring whether a model claim is supported by retrieved evidence.
    Use only the retrieved evidence below. Do not use outside knowledge.
    Reply with only one number from 0.00 to 1.00.
    0.00 means the claim is not supported. 1.00 means the claim is fully supported.
    Do not repeat these instructions and do not write words before or after the number.
  user_template: |
    Retrieved evidence:
    {evidence}

    Question:
    {question}

    Claim:
    {claim}

    Faithfulness score:

# Abstain message lives here
uncertainty:
  notes: "Phase 10 — combined retrieval+verification confidence; ANSWER | ABSTAIN gate."
  abstention_message: >-
    I cannot answer reliably because supporting evidence is insufficient.
  method: "mean_retrieval_verification"
```


# 4. Calibration & Threshold

This is the section for: **“Where did 0.65 come from?”**

The short answer: T was chosen on the **40-question FinQA DEV calibration set**, then locked. The frozen **140-question TEST set** was scored after that. T was not tuned on TEST.

The rule in `V2/src/calibration/data.py` is:

1. Use the 40-question DEV / calibration set.
2. Test candidate confidence thresholds.
3. Require coverage >= 0.50.
4. Choose the threshold with the highest selective accuracy.
5. If there is a tie, use the lower threshold.
6. Lock the selected threshold in `threshold.lock.json`.
7. Only then evaluate the frozen 140-question TEST set.

Locked result: **T = 0.65**.


## `select.py`

**File:** `V2/src/calibration/select.py`

**What this file does**  
It tries many possible T values on the DEV UQ cases and picks one using the pre-registered rule.

**What goes in**  
40 DEV cases, each with a confidence score and a draft answer. Correctness is `numeric_match` against `program_answer`.

**What comes out**  
The selected threshold, plus coverage and selective accuracy at that T.

**What I can say in the viva**  
`select_threshold()` keeps every T with coverage at least 0.50, then takes the highest selective accuracy. A tie takes the lowest T (`-row["threshold"]` in the `max(...)` key). This never looks at the frozen 140.


In [ ]:
"""Pre-registered DEV-only threshold selection. Never inspect the frozen 140."""

from __future__ import annotations

from typing import Any

from src.calibration.data import COVERAGE_FLOOR, SELECTION_RULE, TIE_BREAK
from src.evaluation.numeric import numeric_match

GRID_STEP = 0.01


def draft_text(case: dict[str, Any]) -> str:
    cfg = case.get("configuration") or {}
    draft = cfg.get("draft_answer")
    if draft:
        return str(draft)
    return str(case.get("answer") or "")


def case_to_point(case: dict[str, Any]) -> dict[str, Any]:
    confidence = case.get("confidence")
    gold = case.get("reference_answer")
    predicted = draft_text(case)
    # Was the draft number correct?
    correct = numeric_match(predicted, gold) if confidence is not None else False
    return {
        "question_id": case.get("question_id"),
        "confidence": None if confidence is None else float(confidence),
        "correct": bool(correct),
        "gold": gold,
        "draft": predicted,
        "smoke_decision": case.get("decision"),
    }


# Test candidate thresholds
def _candidate_thresholds(confidences: list[float]) -> list[float]:
    grid = [round(i * GRID_STEP, 2) for i in range(0, 101)]
    observed = sorted({round(float(c), 4) for c in confidences})
    merged = sorted(set(grid + observed))
    return merged


def metrics_at_threshold(points: list[dict[str, Any]], threshold: float) -> dict[str, Any]:
    usable = [p for p in points if p.get("confidence") is not None]
    n = len(usable)
    # ANSWER if confidence >= T
    answered = [p for p in usable if float(p["confidence"]) >= threshold]
    abstained = n - len(answered)
    n_answer = len(answered)
    n_correct = sum(1 for p in answered if p["correct"])
    # How often we answered
    coverage = (n_answer / n) if n else 0.0
    # Accuracy among answered cases
    selective_accuracy = (n_correct / n_answer) if n_answer else 0.0
    return {
        "threshold": float(threshold),
        "n": n,
        "n_answer": n_answer,
        "n_abstain": abstained,
        "n_correct_answered": n_correct,
        "coverage": coverage,
        "selective_accuracy": selective_accuracy,
        "meets_coverage_floor": coverage >= COVERAGE_FLOOR,
    }


def sweep_thresholds(points: list[dict[str, Any]]) -> list[dict[str, Any]]:
    confidences = [float(p["confidence"]) for p in points if p.get("confidence") is not None]
    if not confidences:
        raise ValueError("No calibration confidences to sweep")
    return [metrics_at_threshold(points, t) for t in _candidate_thresholds(confidences)]


# Select the best threshold
def select_threshold(points: list[dict[str, Any]]) -> dict[str, Any]:
    """Maximise selective accuracy among T with coverage >= 0.50; tie → lowest T."""
    # Try many possible T values
    curve = sweep_thresholds(points)
    # Keep T with coverage at least 0.50
    feasible = [row for row in curve if row["meets_coverage_floor"]]
    if not feasible:
        return {
            "selected": False,
            "threshold": None,
            "rule": SELECTION_RULE,
            "coverage_floor": COVERAGE_FLOOR,
            "tie_break": TIE_BREAK,
            "reason": f"No threshold achieved coverage >= {COVERAGE_FLOOR}",
            "curve": curve,
        }
    # Pick the best T
    best = max(feasible, key=lambda row: (row["selective_accuracy"], -row["threshold"]))
    return {
        "selected": True,
        "threshold": best["threshold"],
        "rule": SELECTION_RULE,
        "coverage_floor": COVERAGE_FLOOR,
        "tie_break": TIE_BREAK,
        "coverage": best["coverage"],
        "selective_accuracy": best["selective_accuracy"],
        "n_answer": best["n_answer"],
        "n_abstain": best["n_abstain"],
        "n": best["n"],
        "curve": curve,
    }


## `data.py`

**File:** `V2/src/calibration/data.py`

**What this file does**  
Holds the frozen calibration rules: 40 DEV questions, coverage floor 0.50, and the tie-break.

**What goes in**  
The calibration CSV (`data/calibration/calibration_questions.csv`).

**What comes out**  
The constants that `select.py` uses, plus a leakage check so TEST ids cannot enter calibration.

**What I can say in the viva**  
If asked “who decided the rule?”, this is the file. `COVERAGE_FLOOR = 0.50`, `TIE_BREAK = "lowest_threshold"`.


In [ ]:
"""Phase 13: load the frozen FinQA DEV calibration set (never the frozen 140)."""

from __future__ import annotations

import csv
import json
from pathlib import Path
from typing import Any

from src.config import ExperimentConfig, get_path, load_experiment_config, project_root
from src.data.select_calibration import rows_fingerprint
from src.run.subset import load_frozen_question_rows

# 40 DEV questions
CALIBRATION_N = 40
# Need coverage at least 0.50
COVERAGE_FLOOR = 0.50
# Highest selective accuracy
SELECTION_RULE = "max_selective_accuracy_coverage_ge_0.50"
# Lower T wins a tie
TIE_BREAK = "lowest_threshold"


def calibration_csv(config: ExperimentConfig | None = None) -> Path:
    cfg = config or load_experiment_config()
    dataset = cfg.section("dataset")
    rel = str(dataset.get("frozen_calibration_set") or "data/calibration/calibration_questions.csv")
    path = (project_root() / rel).resolve()
    if not path.is_file():
        path = get_path(cfg, "data_calibration") / "calibration_questions.csv"
    return path


def calibration_manifest_path(config: ExperimentConfig | None = None) -> Path:
    cfg = config or load_experiment_config()
    dataset = cfg.section("dataset")
    rel = str(dataset.get("calibration_manifest") or "data/calibration/calibration_manifest.json")
    return (project_root() / rel).resolve()


def load_calibration_questions(
    *,
    n: int = CALIBRATION_N,
    config: ExperimentConfig | None = None,
    csv_path: Path | None = None,
) -> list[dict[str, str]]:
    if n < 1:
        raise ValueError("Calibration n must be >= 1")
    if n > CALIBRATION_N:
        raise ValueError(
            f"Phase 13 calibration is capped at {CALIBRATION_N} DEV questions. "
            "Do not use the frozen 140 or start the 420-case benchmark here."
        )
    path = csv_path or calibration_csv(config)
    rows: list[dict[str, str]] = []
    with path.open("r", encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle):
            qid = str(row.get("id") or "").strip()
            question = str(row.get("question") or "").strip()
            split = str(row.get("split") or "dev").strip().lower()
            if not qid or not question:
                continue
            if split and split != "dev":
                raise ValueError(f"Calibration row {qid} has split={split!r}; expected dev")
            rows.append(
                {
                    "id": qid,
                    "question": question,
                    "program_answer": str(row.get("program_answer") or ""),
                    "original_answer": str(row.get("original_answer") or ""),
                    "file_name": str(row.get("file_name") or ""),
                    "split": "dev",
                }
            )
            if len(rows) >= n:
                break
    if len(rows) < n:
        raise ValueError(f"Calibration CSV has {len(rows)} rows; need {n}")
    return rows


def frozen_test_ids(config: ExperimentConfig | None = None) -> set[str]:
    return {row["id"] for row in load_frozen_question_rows()}


# Make sure TEST was not used
def assert_no_test_leakage(questions: list[dict[str, str]], *, config: ExperimentConfig | None = None) -> None:
    test_ids = frozen_test_ids(config)
    leaked = [row["id"] for row in questions if row["id"] in test_ids]
    if leaked:
        raise RuntimeError(f"Calibration set leaks frozen test IDs: {leaked[:5]}")
    bad_prefix = [row["id"] for row in questions if not str(row["id"]).startswith("finqa_dev_")]
    if bad_prefix:
        raise RuntimeError(f"Calibration IDs must be FinQA DEV, got: {bad_prefix[:5]}")


def load_calibration_manifest(path: Path | None = None) -> dict[str, Any]:
    dest = path or calibration_manifest_path()
    return json.loads(dest.read_text(encoding="utf-8"))


def verify_calibration_subset(questions: list[dict[str, str]]) -> dict[str, Any]:
    manifest = load_calibration_manifest()
    ids = [row["id"] for row in questions]
    expected = list(manifest.get("selected_ids") or [])[: len(ids)]
    if expected and ids != expected:
        raise ValueError("Calibration question IDs do not match the frozen Phase 5 manifest order")
    expected_sha = str(manifest.get("selected_ids_sha256") or "")
    if len(ids) == CALIBRATION_N and expected_sha and rows_fingerprint(questions) != expected_sha:
        raise ValueError("Calibration ID SHA-256 does not match the Phase 5 manifest")
    assert_no_test_leakage(questions)
    return manifest


## `lock.py`

**File:** `V2/src/calibration/lock.py`

**What this file does**  
Writes or refuses `threshold.lock.json`, and later loads it without changing T.

**What goes in**  
A finished DEV calibration run (official lock needs llama_cpp/CUDA and 40 cases). Later phases only read the lock.

**What comes out**  
The lock file, or an error if someone tries to use TEST, mock, or a different T.

**What I can say in the viva**  
The locked value is 0.65. `used_frozen_test_140` must be false. Later code reads this file instead of guessing T.


In [ ]:
"""Write or refuse ``threshold.lock.json``. Official lock requires a real DEV run."""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

from src.calibration.data import CALIBRATION_N, assert_no_test_leakage
from src.calibration.select import case_to_point, select_threshold
from src.config import get_path, load_experiment_config, project_root
from src.run.store import utc_now
from src.run.subset import ids_sha256

LOCK_FILENAME = "threshold.lock.json"
CANDIDATE_FILENAME = "threshold.candidate.json"
OFFICIAL_BACKENDS = {"llama_cpp", "transformers"}
# Locked T = 0.65
EXPECTED_LOCKED_THRESHOLD = 0.65


def lock_path() -> Path:
    return get_path(load_experiment_config(), "results_config") / LOCK_FILENAME


def candidate_path() -> Path:
    return get_path(load_experiment_config(), "results_config") / CANDIDATE_FILENAME


def load_cases(jsonl_path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with jsonl_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            text = line.strip()
            if not text:
                continue
            rows.append(json.loads(text))
    return rows


def official_lock_allowed(*, backend: str, device: str | None, n_completed: int) -> tuple[bool, str]:
    if str(backend).lower() in {"mock", "test"}:
        return False, "Mock backend cannot lock the official threshold."
    if str(backend).lower() not in OFFICIAL_BACKENDS:
        return False, f"Backend {backend!r} is not an official lock backend ({sorted(OFFICIAL_BACKENDS)})."
    if device in {None, "mps_capable_host"}:
        return False, "Official lock requires a CUDA Colab-style device, not Mac MPS."
    if n_completed < CALIBRATION_N:
        return False, f"Official lock requires {CALIBRATION_N} completed DEV cases (got {n_completed})."
    return True, "ok"


def build_lock_payload(
    cases: list[dict[str, Any]],
    *,
    run_id: str,
    backend: str,
    device: str | None,
    gpu: Any,
    git_commit: str | None,
    official: bool,
) -> dict[str, Any]:
    question_ids = [str(case.get("question_id")) for case in cases]
    fake_rows = [{"id": qid} for qid in question_ids]
    # Make sure TEST was not used
    assert_no_test_leakage(fake_rows)
    points = [case_to_point(case) for case in cases]
    # Choose T on DEV only
    selection = select_threshold(points)
    curve = selection.pop("curve")
    payload = {
        "phase": 13,
        "mode": "calibration",
        "locked": bool(official and selection.get("selected")),
        "threshold": selection.get("threshold"),
        "rule": selection.get("rule"),
        "coverage_floor": selection.get("coverage_floor"),
        "tie_break": selection.get("tie_break"),
        "coverage": selection.get("coverage"),
        "selective_accuracy": selection.get("selective_accuracy"),
        "n": selection.get("n"),
        "n_answer": selection.get("n_answer"),
        "n_abstain": selection.get("n_abstain"),
        "selected": selection.get("selected"),
        "reason": selection.get("reason"),
        "source_split": "dev",
        "used_frozen_test_140": False,
        "n_cases": len(cases),
        "question_ids": question_ids,
        "question_ids_sha256": ids_sha256(question_ids),
        "run_id": run_id,
        "backend": backend,
        "device": device,
        "gpu": gpu,
        "git_commit": git_commit,
        "architecture": "multi_agent_uq",
        "recorded_at_utc": utc_now(),
        "curve": curve,
    }
    if official and payload["locked"]:
        payload["threshold_note"] = "LOCKED on FinQA DEV calibration — not tuned on the frozen 140"
    else:
        payload["threshold_note"] = "candidate only — NOT LOCKED"
        payload["locked"] = False
    return payload


def write_threshold_files(payload: dict[str, Any]) -> dict[str, str]:
    out_dir = get_path(load_experiment_config(), "results_config")
    out_dir.mkdir(parents=True, exist_ok=True)
    cand = out_dir / CANDIDATE_FILENAME
    cand.write_text(json.dumps(payload, indent=2, default=str) + "\n", encoding="utf-8")
    written = {"candidate": _rel(cand)}
    # Lock the final threshold
    if payload.get("locked"):
        # Write the lock file
        lock = out_dir / LOCK_FILENAME
        lock.write_text(json.dumps(payload, indent=2, default=str) + "\n", encoding="utf-8")
        written["lock"] = _rel(lock)
    return written


def _rel(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(project_root()))
    except ValueError:
        return str(path)


# Read the lock, do not change T
def load_official_lock(path: Path | None = None) -> dict[str, Any]:
    """Read-only load of the Phase 13 DEV lock. Never writes or recalibrates."""
    dest = Path(path) if path is not None else lock_path()
    if not dest.is_file():
        raise FileNotFoundError(
            f"Official threshold lock missing at {dest}. Run Phase 13 Colab lock first."
        )
    payload = json.loads(dest.read_text(encoding="utf-8"))
    if payload.get("locked") is not True:
        raise RuntimeError("threshold.lock.json is not locked. Phase 14 will not invent a threshold.")
    if payload.get("used_frozen_test_140") is True:
        raise RuntimeError("Lock claims the frozen 140 was used. Refusing to run Phase 14.")
    if str(payload.get("source_split") or "") != "dev":
        raise RuntimeError("Official lock must have source_split=dev.")
    if int(payload.get("phase") or 0) < 13:
        raise RuntimeError("Official lock phase must be >= 13.")


In [ ]:
    threshold = float(payload.get("threshold"))
    # Must still be 0.65
    if abs(threshold - EXPECTED_LOCKED_THRESHOLD) > 1e-9:
        raise RuntimeError(
            f"Phase 14 requires locked T={EXPECTED_LOCKED_THRESHOLD}, found {threshold}. "
            "Do not recalibrate or modify the threshold."
        )
    payload["threshold"] = threshold
    return payload


def locked_threshold(path: Path | None = None) -> float:
    return float(load_official_lock(path)["threshold"])


## Threshold configuration

The live lock file is `V2/results/config/threshold.lock.json`.

That JSON is the real lock. The important fields are:

| Field | Value in the lock file |
|---|---|
| `locked` | `true` |
| `threshold` | `0.65` |
| `rule` | `max_selective_accuracy_coverage_ge_0.50` |
| `coverage_floor` | `0.5` |
| `tie_break` | `lowest_threshold` |
| `coverage` | `0.55` (22/40 answered on DEV) |
| `selective_accuracy` | `0.54545...` (12/22) |
| `source_split` | `dev` |
| `used_frozen_test_140` | `false` |
| `n` | `40` |

On the DEV curve inside that same file, T = 0.65 is the best feasible point. T = 0.66 is worse on DEV (selective accuracy 11/21 ≈ 0.5238). So 0.66 is **not** a second research threshold.

In `V2/config/experiment.yaml`, `uncertainty.confidence_threshold` stays `null`. Official T comes from the lock file.

The Streamlit 0.65–0.66 warning is display-only. It lives in `uq_ui_confidence_overlay()` in `V2/src/rag/live.py`. It does not change ANSWER / ABSTAIN.


## `threshold.lock.json`

**File:** `V2/results/config/threshold.lock.json`

This is the actual lock used by later phases and by Streamlit (`load_official_lock()` / `resolve_live_locked_threshold()`).

The JSON below is copied from that file. The long 140-row DEV `curve` is not pasted in full. The selected **T = 0.65** row and the unused **0.66** row are the real objects from that curve.

**What I can say in the viva**  
`threshold` is 0.65, `source_split` is `dev`, `used_frozen_test_140` is false. That is where 0.65 comes from.


```json
{
  "phase": 13,
  "mode": "calibration",
  "locked": true,
  "threshold": 0.65,
  "rule": "max_selective_accuracy_coverage_ge_0.50",
  "coverage_floor": 0.5,
  "tie_break": "lowest_threshold",
  "coverage": 0.55,
  "selective_accuracy": 0.5454545454545454,
  "n": 40,
  "n_answer": 22,
  "n_abstain": 18,
  "selected": true,
  "reason": null,
  "source_split": "dev",
  "used_frozen_test_140": false,
  "n_cases": 40,
  "question_ids": [
    "finqa_dev_130",
    "finqa_dev_142",
    "finqa_dev_178",
    "finqa_dev_198",
    "finqa_dev_233",
    "finqa_dev_280",
    "finqa_dev_298",
    "finqa_dev_304",
    "finqa_dev_320",
    "finqa_dev_343",
    "finqa_dev_359",
    "finqa_dev_38",
    "finqa_dev_400",
    "finqa_dev_428",
    "finqa_dev_441",
    "finqa_dev_451",
    "finqa_dev_460",
    "finqa_dev_490",
    "finqa_dev_492",
    "finqa_dev_50",
    "finqa_dev_509",
    "finqa_dev_542",
    "finqa_dev_562",
    "finqa_dev_578",
    "finqa_dev_579",
    "finqa_dev_58",
    "finqa_dev_592",
    "finqa_dev_623",
    "finqa_dev_638",
    "finqa_dev_654",
    "finqa_dev_661",
    "finqa_dev_682",
    "finqa_dev_695",
    "finqa_dev_722",
    "finqa_dev_734",
    "finqa_dev_77",
    "finqa_dev_787",
    "finqa_dev_813",
    "finqa_dev_819",
    "finqa_dev_880"
  ],
  "question_ids_sha256": "da2126411f3025570293725bc93e5bd8d118dfbc2c706ec53c51eddcf38c4853",
  "run_id": "phase13_20260826T192003Z_7bcd6ed3",
  "backend": "llama_cpp",
  "device": "cuda",
  "gpu": {
    "available": true,
    "name": "Tesla T4",
    "vram_total_mb": 15360.0,
    "vram_free_mb": 14910.0,
    "driver_version": "580.82.07"
  },
  "git_commit": "19368f1a53aacc4c4396b6d648e0722cc1c802ea",
  "architecture": "multi_agent_uq",
  "recorded_at_utc": "2026-08-26T19:36:59.876900+00:00",
  "threshold_note": "LOCKED on FinQA DEV calibration \u2014 not tuned on the frozen 140",
  "curve_selected_row": {
    "threshold": 0.65,
    "n": 40,
    "n_answer": 22,
    "n_abstain": 18,
    "n_correct_answered": 12,
    "coverage": 0.55,
    "selective_accuracy": 0.5454545454545454,
    "meets_coverage_floor": true
  },
  "curve_row_at_0_66_not_selected": {
    "threshold": 0.66,
    "n": 40,
    "n_answer": 21,
    "n_abstain": 19,
    "n_correct_answered": 11,
    "coverage": 0.525,
    "selective_accuracy": 0.5238095238095238,
    "meets_coverage_floor": true
  },
  "curve_note": "The real file also has a DEV curve of 140 candidate rows. Only the selected 0.65 row and the 0.66 row are copied here so the notebook stays readable."
}
```


## `live.py` — 0.66 UI warning

**File:** `V2/src/rag/live.py`  
Functions: `_confidence_in_lock_hundredths()`, `uq_ui_confidence_overlay()`, `resolve_live_locked_threshold()`.

This is not a second threshold. Streamlit shows “Moderate confidence — verify supporting evidence.” when UQ **ANSWER** confidence is still in the 0.65 hundredths band (`0.65 ≤ c < 0.66`). The stored decision stays ANSWER. T stays 0.65.

The docstring says it plainly: *UI proximity only. Not a second research threshold. Not written to the lock file.*


In [ ]:
UI_ABSTAIN_LOW_CONFIDENCE = "ABSTAIN — Low confidence"
UI_MODERATE_CONFIDENCE_WARNING = "Moderate confidence — verify supporting evidence."
UI_CONFIDENCE_WARNING_NOTE = (
    "Warning is a user-facing confidence indicator and does not alter the research decision rule."
)


# 0.65 to 0.66 is display only
def _confidence_in_lock_hundredths(confidence: float, locked_t: float) -> bool:
    """True when confidence is still in the locked T hundredths band (e.g. 0.65 ≤ c < 0.66).

    UI proximity only. Not a second research threshold. Not written to the lock file.
    """
    return locked_t <= confidence < locked_t + 0.01


def uq_ui_confidence_overlay(result: RAGCaseResult) -> dict[str, Any]:
    """Streamlit-only UQ captions. Does not mutate decision, confidence, or stored results."""
    empty = {
        "show": False,
        "decision_heading": None,
        "warning": None,
        "note": None,
    }
    if result.architecture != ARCHITECTURE_MULTI_AGENT_UQ:
        return empty
    note = UI_CONFIDENCE_WARNING_NOTE
    confidence = resolve_displayed_confidence(result)
    if confidence is None:
        return {
            "show": True,
            "decision_heading": result.decision,
            "warning": None,
            "note": note,
        }
    heading = result.decision
    warning = None
    if result.decision == "ABSTAIN":
        heading = UI_ABSTAIN_LOW_CONFIDENCE
    elif result.decision == "ANSWER" and result.threshold is not None:
        locked_t = float(result.threshold)
        if confidence >= locked_t and _confidence_in_lock_hundredths(confidence, locked_t):
            # Show a warning, do not change T
            warning = UI_MODERATE_CONFIDENCE_WARNING
    return {
        "show": True,
        "decision_heading": heading,
        "warning": warning,
        "note": note,
    }


# Read locked T = 0.65
def resolve_live_locked_threshold() -> float:
    """Official T from threshold.lock.json. Does not retune or read yaml smoke_threshold."""
    from src.calibration.lock import EXPECTED_LOCKED_THRESHOLD, load_official_lock

    lock = load_official_lock()
    threshold = float(lock["threshold"])
    if abs(threshold - EXPECTED_LOCKED_THRESHOLD) > 1e-9:
        raise RuntimeError(
            f"Live artefact requires locked T={EXPECTED_LOCKED_THRESHOLD}, found {threshold}."
        )
    if lock.get("used_frozen_test_140") is True:
        raise RuntimeError("Lock claims the frozen 140 was used. Refusing live demo.")
    if str(lock.get("source_split") or "") != "dev":
        raise RuntimeError("Official lock must have source_split=dev.")
    return threshold


# 5. Evaluation & Metrics

These files score the saved 420 cases. They do **not** generate new RAG answers.

The judge is a **custom Qwen3-8B LLM-as-judge / RAGAS-inspired judge**. It is **not** official RAGAS.

| File | What it measures |
|---|---|
| `numeric.py` | Is the predicted number close to FinQA `program_answer`? |
| `metrics.py` | One saved case: correctness, coverage, selective accuracy, unsupported-emitted, context P/R, numeric context recall, confidence, latency, token-overlap faithfulness |
| `judge.py` | Post-hoc 0–1 faithfulness / support score from Qwen |

| Metric in the code | What it means | Where it is |
|---|---|---|
| correctness | Displayed number matches gold (`answer_correctness`) | `numeric_match()` → `score_case()` |
| coverage | Fraction of cases with `decision == "ANSWER"` | `aggregate_architecture()` |
| answered-case / selective accuracy | Correctness among ANSWER only | `selective_accuracy` in `aggregate_architecture()` |
| unsupported-emitted | ANSWERED and the displayed number is wrong | `unsupported_emitted` in `score_case()` |
| context precision | Retrieved chunks that match gold PDF / `context_id` | `context_precision()` |
| context recall | Did any retrieved chunk match the gold PDF? | `context_recall()` |
| numeric context recall | Does the retrieved text contain the gold number? | `context_recall_numeric` |
| confidence | Stored UQ score C = (R+V)/2 | `score_case()` copies `case["confidence"]` |
| latency | Seconds stored on the case | `latency_seconds` |
| faithfulness (CPU) | Token overlap of claim vs evidence | `token_overlap()` in `metrics.py` |
| faithfulness (judge) | Qwen 0–1 support score | `parsed_faithfulness_score` in `judge.py` |


## `numeric.py`

**File:** `V2/src/evaluation/numeric.py`

**What this file measures**  
Primary correctness: extract numbers from text and test whether any predicted number matches gold `program_answer` (relative tolerance 0.01).

**What goes in**  
Predicted text and gold `program_answer`.

**What comes out**  
`True` or `False`.

**What I can say in the viva**  
RQ1 accuracy in this project means this numeric match. It is not BLEU and it is not official RAGAS.


In [ ]:
"""Numeric match against FinQA ``program_answer`` (calibration / later metrics)."""

from __future__ import annotations

import math
import re

_NUMBER_RE = re.compile(
    r"(?<![A-Za-z])[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?(?:\s*%)?",
    re.IGNORECASE,
)


def parse_numbers(text: str | None) -> list[float]:
    """Extract numeric tokens. Values with % are stored as given and as /100."""
    if not text:
        return []
    values: list[float] = []
    # Pull numbers out of the text
    for match in _NUMBER_RE.finditer(str(text)):
        raw = match.group(0).strip()
        percent = raw.endswith("%")
        cleaned = raw.replace(",", "").replace("%", "").strip()
        try:
            number = float(cleaned)
        except ValueError:
            continue
        values.append(number)
        if percent:
            values.append(number / 100.0)
        else:
            values.append(number / 100.0)
            values.append(number * 100.0)
    return values


def numeric_match(
    predicted: str | None,
    gold: str | None,
    *,
    rel_tol: float = 0.01,
    abs_tol: float = 1e-4,
) -> bool:
    """True if any number in ``predicted`` matches ``gold`` within tolerance."""
    if gold is None or str(gold).strip() == "":
        return False
    gold_values = parse_numbers(str(gold))
    if not gold_values:
        return False
    gold_number = gold_values[0]
    for predicted_number in parse_numbers(predicted):
        # Match predicted number to gold
        if math.isclose(predicted_number, gold_number, rel_tol=rel_tol, abs_tol=abs_tol):
            return True
    return False


## `metrics.py`

**File:** `V2/src/evaluation/metrics.py`

**What this file measures**  
CPU scoring of one saved architecture–question case, then a per-architecture summary.

**What goes in**  
A saved Phase 15 case and the gold row from the frozen CSV.

**What the main outputs mean**

- `answer_correctness` — displayed answer number is right
- `answer_correctness_claim` — draft/claim number is right (for UQ ABSTAIN this is the hidden draft)
- `answered` / coverage — `decision == "ANSWER"`
- `selective_accuracy` — correctness among answered cases only
- `unsupported_emitted` — we answered, but the displayed number was wrong (not a hallucination label)
- `context_precision` / `context_recall` — did retrieval hit the gold PDF / `context_id`?
- `context_recall_numeric` — does the retrieved text contain the gold number?
- `confidence` / `latency_seconds` — copied from the saved case
- `faithfulness` — CPU token overlap, not the judge score

**What I can say in the viva**  
For UQ abstentions, the claim is the hidden draft, not the abstention sentence. `unsupported_emitted` means “answered and numerically wrong”.


In [ ]:
"""Phase 16 CPU metrics from saved RAG cases. No LLM / GPU / new generation."""

from __future__ import annotations

from typing import Any

from src.evaluation.numeric import numeric_match
from src.rag.text_utils import token_overlap

ARCHITECTURE_UQ = "multi_agent_uq"


def _norm_path(value: str | None) -> str:
    return str(value or "").replace("\\", "/").strip().lower()


def files_match(left: str | None, right: str | None) -> bool:
    a, b = _norm_path(left), _norm_path(right)
    if not a or not b:
        return False
    return a == b or a.endswith("/" + b) or b.endswith("/" + a) or a.endswith(b) or b.endswith(a)


def chunk_is_relevant(chunk: dict[str, Any], gold: dict[str, str]) -> bool:
    gold_file = gold.get("file_name") or ""
    gold_ctx = str(gold.get("context_id") or "").strip()
    chunk_file = str(chunk.get("file_name") or "")
    chunk_ctx = str(chunk.get("context_id") or "").strip()
    if gold_ctx and chunk_ctx and gold_ctx == chunk_ctx:
        return True
    return files_match(chunk_file, gold_file)


def evidence_text(chunks: list[dict[str, Any]]) -> str:
    parts: list[str] = []
    for chunk in chunks:
        text = chunk.get("text") or chunk.get("content") or ""
        if text:
            parts.append(str(text))
    return "\n".join(parts)


def scored_claim_text(case: dict[str, Any]) -> str:
    """Text used for correctness/faithfulness of the model's claim.

    UQ ABSTAIN replaces the displayed answer with the abstention template;
    the draft is the claim that would have been emitted.
    """
    cfg = case.get("configuration") or {}
    draft = cfg.get("draft_answer")
    if case.get("architecture") == ARCHITECTURE_UQ and draft:
        return str(draft)
    return str(case.get("answer") or "")


def displayed_text(case: dict[str, Any]) -> str:
    return str(case.get("answer") or "")


def context_precision(chunks: list[dict[str, Any]], gold: dict[str, str]) -> float:
    if not chunks:
        return 0.0
    relevant = sum(1 for chunk in chunks if chunk_is_relevant(chunk, gold))
    return relevant / len(chunks)


def context_recall(chunks: list[dict[str, Any]], gold: dict[str, str]) -> float:
    if not chunks:
        return 0.0
    return 1.0 if any(chunk_is_relevant(chunk, gold) for chunk in chunks) else 0.0


def score_case(case: dict[str, Any], gold: dict[str, str]) -> dict[str, Any]:
    """Score one saved architecture–question case. CPU only."""
    chunks = list(case.get("retrieved_evidence") or [])
    gold_program = gold.get("program_answer") or case.get("reference_answer")
    gold_original = gold.get("original_answer") or ""
    claim = scored_claim_text(case)
    displayed = displayed_text(case)
    evidence = evidence_text(chunks)
    decision = str(case.get("decision") or "ANSWER")
    # Did we ANSWER or ABSTAIN?
    answered = decision == "ANSWER"

    # Check if the number is correct
    correct_claim = numeric_match(claim, gold_program)
    correct_displayed = numeric_match(displayed, gold_program)
    correct_original = numeric_match(claim, gold_original) if gold_original else False

    # How much the answer overlaps evidence
    faithfulness = token_overlap(claim, evidence)
    stored_verify = case.get("verification_result") if isinstance(case.get("verification_result"), dict) else {}
    # Did we retrieve the right PDF?
    precision = context_precision(chunks, gold)
    recall = context_recall(chunks, gold)
    # Numeric context recall
    recall_numeric = numeric_match(evidence, gold_program)

    # Answered but the number was wrong
    unsupported_emitted = bool(answered and not correct_displayed)

    return {
        "case_key": case.get("case_key") or f"{case.get('architecture')}:{case.get('question_id')}",
        "run_id": case.get("run_id"),
        "question_id": case.get("question_id"),
        "architecture": case.get("architecture"),
        "decision": decision,
        "answered": answered,
        # Stored confidence
        "confidence": case.get("confidence"),
        "threshold": case.get("threshold"),
        "n_evidence": len(chunks),
        "gold_program_answer": gold_program,
        "gold_file_name": gold.get("file_name"),
        "gold_context_id": gold.get("context_id"),
        "answer_correctness": int(correct_displayed),
        "answer_correctness_claim": int(correct_claim),
        "answer_correctness_original_answer": int(correct_original),
        "faithfulness": faithfulness,
        "faithfulness_stored_verification_score": stored_verify.get("verification_score"),
        "faithfulness_stored_lexical_score": stored_verify.get("lexical_score"),
        "context_precision": precision,
        "context_recall": recall,
        "context_recall_numeric": int(recall_numeric),
        "unsupported_emitted": int(unsupported_emitted),
        # How long the case took
        "latency_seconds": case.get("latency_seconds"),
        "backend": case.get("backend"),
        "device": case.get("device"),
        "gpu": case.get("gpu"),
        "model": case.get("model"),
        "quantisation": case.get("quantisation"),
        "error": case.get("error"),
        "used_llm_inference": False,
        "used_gpu": False,
    }


def mean(values: list[float]) -> float | None:
    if not values:
        return None
    return sum(values) / len(values)


def aggregate_architecture(rows: list[dict[str, Any]]) -> dict[str, Any]:
    n = len(rows)
    answered = [r for r in rows if r.get("answered")]
    abstained = [r for r in rows if not r.get("answered")]


In [ ]:
    n_answer = len(answered)
    n_abstain = len(abstained)
    n_correct_displayed = sum(int(r.get("answer_correctness") or 0) for r in rows)
    n_correct_claim = sum(int(r.get("answer_correctness_claim") or 0) for r in rows)
    n_correct_answered = sum(int(r.get("answer_correctness") or 0) for r in answered)
    n_unsupported = sum(int(r.get("unsupported_emitted") or 0) for r in rows)
    confidences = [float(r["confidence"]) for r in rows if r.get("confidence") is not None]
    stored_v = [
        float(r["faithfulness_stored_verification_score"])
        for r in rows
        if r.get("faithfulness_stored_verification_score") is not None
    ]
    return {
        "n": n,
        "n_answer": n_answer,
        "n_abstain": n_abstain,
        # Check coverage
        "coverage": (n_answer / n) if n else 0.0,
        "abstention_rate": (n_abstain / n) if n else 0.0,
        "answer_correctness": (n_correct_displayed / n) if n else 0.0,
        "answer_correctness_claim": (n_correct_claim / n) if n else 0.0,
        # Accuracy among answered cases
        "selective_accuracy": (n_correct_answered / n_answer) if n_answer else None,
        "n_correct_displayed": n_correct_displayed,
        "n_correct_claim": n_correct_claim,
        "n_correct_answered": n_correct_answered,
        "unsupported_emitted_rate": (n_unsupported / n) if n else 0.0,
        "faithfulness": mean([float(r["faithfulness"]) for r in rows]),
        "faithfulness_stored_verification_score": mean(stored_v),
        "context_precision": mean([float(r["context_precision"]) for r in rows]),
        "context_recall": mean([float(r["context_recall"]) for r in rows]),
        "context_recall_numeric": mean([float(r["context_recall_numeric"]) for r in rows]),
        "mean_confidence": mean(confidences),
        "mean_latency_seconds": mean(
            [float(r["latency_seconds"]) for r in rows if r.get("latency_seconds") is not None]
        ),
    }


## `judge.py`

**File:** `V2/src/evaluation/judge.py`

**What this file measures**  
A **custom Qwen3-8B LLM-as-judge / RAGAS-inspired** faithfulness / support score. It is **not** the official RAGAS library.

The label in the code is:

`LLM-as-judge faithfulness (Qwen3-8B, custom/RAGAS-inspired)`

**What goes in**  
A saved case: question, retrieved chunks, and the claim. UQ uses `draft_answer`, not the abstention template. Gold answers and gold context are not put in the prompt.

**How the score is produced**

1. `build_judge_prompt()` puts evidence, question and claim into the judge template.
2. `prompt_contains_forbidden()` checks that gold did not leak in.
3. Qwen generates a short reply (`llm.generate`).
4. `parse_unit_score()` reads one number from 0.00 to 1.00.
5. That number is stored as `parsed_faithfulness_score`.

**What comes out**  
A 0–1 support score for that saved case. This is the RQ2 faithfulness number.


In [ ]:
"""Post-hoc LLM-as-judge faithfulness over saved Phase 15 cases.

Does not run retrieval or any RAG architecture. Does not read FinQA gold context
or the gold answer into the judge prompt.
"""

from __future__ import annotations

import hashlib
from typing import Any

from src.models.types import LLMBackend
from src.rag.schema import ARCHITECTURE_MULTI_AGENT_UQ
from src.rag.text_utils import parse_unit_score

METRIC_LABEL = "LLM-as-judge faithfulness (Qwen3-8B, custom/RAGAS-inspired)"
PROMPT_ID = "phase16_judge_faithfulness_v1"
JUDGE_TEMPERATURE = 0.0
JUDGE_MAX_NEW_TOKENS = 32
JUDGE_N_CTX = 4096

JUDGE_SYSTEM = (
    "You are scoring whether a model claim is supported by retrieved evidence. "
    "Use only the retrieved evidence below. Do not use outside knowledge. "
    "Reply with only one number from 0.00 to 1.00. "
    "0.00 means the claim is not supported. 1.00 means the claim is fully supported. "
    "Do not repeat these instructions and do not write words before or after the number."
)

JUDGE_USER_TEMPLATE = """Retrieved evidence:
{evidence}

Question:
{question}

Claim:
{claim}

Faithfulness score:"""


def prompt_hash() -> str:
    payload = f"{PROMPT_ID}\n{JUDGE_SYSTEM}\n{JUDGE_USER_TEMPLATE}"
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def format_retrieved_evidence(chunks: list[dict[str, Any]]) -> str:
    """Concatenate retrieved chunk text. Does not use FinQA gold context."""
    parts: list[str] = []
    for i, chunk in enumerate(chunks, start=1):
        text = str(chunk.get("text") or chunk.get("content") or "").strip()
        file_name = str(chunk.get("file_name") or "").strip()
        header = f"[{i}] file={file_name}" if file_name else f"[{i}]"
        parts.append(f"{header}\n{text}" if text else header)
    return "\n\n".join(parts) if parts else "(no retrieved evidence)"


def claim_for_judge(case: dict[str, Any]) -> tuple[str, str]:
    """Return (claim_text, claim_source). UQ uses the draft, not the abstention template."""
    if case.get("architecture") == ARCHITECTURE_MULTI_AGENT_UQ:
        draft = (case.get("configuration") or {}).get("draft_answer")
        return str(draft or ""), "draft_answer"
    return str(case.get("answer") or ""), "answer"


def build_judge_prompt(*, question: str, evidence: str, claim: str) -> str:
    user = JUDGE_USER_TEMPLATE.format(
        evidence=evidence,
        question=(question or "").strip(),
        claim=(claim or "").strip(),
    )
    return f"{JUDGE_SYSTEM.strip()}\n\n{user.strip()}"


def prompt_contains_forbidden(prompt: str, case: dict[str, Any]) -> list[str]:
    """Detect accidental leakage of gold context or gold answers into the judge prompt."""
    hits: list[str] = []
    gold_context = str((case.get("gold_context") or "")).strip()
    if gold_context and len(gold_context) > 40 and gold_context in prompt:
        hits.append("gold_context")
    for key in ("program_answer", "original_answer", "reference_answer"):
        value = str(case.get(key) or "").strip()
        if value and len(value) >= 4 and value in prompt and value not in (case.get("answer") or ""):
            if value not in ((case.get("configuration") or {}).get("draft_answer") or ""):
                if value not in format_retrieved_evidence(list(case.get("retrieved_evidence") or [])):
                    hits.append(key)
    lowered = prompt.lower()
    if "program_answer" in lowered or "gold context" in lowered:
        hits.append("forbidden_label")
    return hits


def judge_one_case(
    case: dict[str, Any],
    llm: LLMBackend,
    *,
    source_raw_sha256: str,
    temperature: float = JUDGE_TEMPERATURE,
    max_new_tokens: int = JUDGE_MAX_NEW_TOKENS,
    n_ctx: int = JUDGE_N_CTX,
    fingerprint: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Score one saved case. Does not retrieve or regenerate the RAG answer."""
    fp = fingerprint or {}
    gpu = fp.get("gpu")
    gpu_name = gpu.get("name") if isinstance(gpu, dict) else gpu
    claim, claim_source = claim_for_judge(case)
    chunks = list(case.get("retrieved_evidence") or [])
    evidence = format_retrieved_evidence(chunks)
    question = str(case.get("question") or "")
    # Build the judge prompt
    prompt = build_judge_prompt(question=question, evidence=evidence, claim=claim)
    # Check gold did not leak in
    leaked = prompt_contains_forbidden(prompt, case)
    record: dict[str, Any] = {
        "case_key": case.get("case_key") or f"{case.get('architecture')}:{case.get('question_id')}",
        "question_id": case.get("question_id"),
        "architecture": case.get("architecture"),
        "source_raw_sha256": source_raw_sha256,
        "claim_source": claim_source,
        "decision": case.get("decision"),
        "judge_model": "Qwen3-8B",
        "judge_metric_label": METRIC_LABEL,
        "backend": getattr(llm, "name", None),
        "device": fp.get("device"),
        "gpu": gpu_name,
        "quantisation": "Q4_K_M" if getattr(llm, "name", "") == "llama_cpp" else getattr(llm, "name", None),
        "n_ctx": n_ctx,
        "temperature": temperature,
        "max_new_tokens": max_new_tokens,
        "prompt_id": PROMPT_ID,
        "prompt_hash": prompt_hash(),
        "raw_judge_output": None,
        "parsed_faithfulness_score": None,
        "parse_failure": False,
        "latency_seconds": None,
        "used_rag_rerun": False,
        "used_gold_context": False,
        "used_gold_answer": False,
        "error": None,
    }
    if leaked:


In [ ]:
        record["error"] = f"judge_prompt_leak:{','.join(leaked)}"
        record["parse_failure"] = True
        return record
    if not claim.strip():
        record["error"] = "empty_claim"
        record["parse_failure"] = True
        return record
    if not any(str(c.get("text") or c.get("content") or "").strip() for c in chunks):
        record["error"] = "empty_retrieved_evidence"
        record["parse_failure"] = True
        return record
    try:
        # Ask Qwen for a 0 to 1 score
        gen = llm.generate(prompt, temperature=temperature, max_new_tokens=max_new_tokens)
    except Exception as exc:  # noqa: BLE001
        record["error"] = str(exc)
        record["parse_failure"] = True
        return record
    raw = gen.text or ""
    record["raw_judge_output"] = raw
    record["latency_seconds"] = gen.latency_seconds
    record["backend"] = gen.backend or record["backend"]
    record["quantisation"] = gen.quantisation or record["quantisation"]
    # Parse the faithfulness score
    parsed = parse_unit_score(raw)
    if parsed is None:
        record["parse_failure"] = True
        record["error"] = "parse_failure"
        return record
    record["parsed_faithfulness_score"] = float(parsed)
    record["parse_failure"] = False
    record["error"] = None
    return record


# 6. Statistical Analysis

This is the final statistics backend. It reads frozen Phase 16 scores. It does **not** rerun RAG, Qwen, the judge, or calibration.

The statistical unit is the **question** (n = 140), paired across the three architectures. Do not treat 420 cases as 420 independent samples.

| Need this in the viva | File | Function |
|---|---|---|
| McNemar test | `V2/src/statistics/tests.py` | `mcnemar_exact()` |
| Spearman correlation | `tests.py` | `spearman_corr()` |
| Mann–Whitney test | `tests.py` | `mannwhitney()` |
| Wilcoxon signed-rank | `tests.py` | `wilcoxon_paired()` |
| Holm correction | `tests.py` | `holm_adjust()` |
| Effect sizes | `tests.py` | `effect_cohens_g`, `cohen_dz()`, `effect_rank_biserial` |
| Assumption checks | `tests.py` / `analysis.py` | `shapiro_wilk()` |
| Which test belongs to which RQ | `V2/src/statistics/analysis.py` | `analyse()` |

`figures.py` and `report.py` only write the already-computed tables and plots. They are not extra tests.


## `constants.py`

**File:** `V2/src/statistics/constants.py`

**What this file does**  
Stores n = 140, α = 0.05, locked T = 0.65, and the judge label.

**What I can say in the viva**  
The judge string here is `LLM-as-judge faithfulness (Qwen3-8B, custom/RAGAS-inspired)`. Not official RAGAS.


In [ ]:
"""Phase 17 constants. Frozen artefact hashes only — do not retune T or freeze files."""

from __future__ import annotations

PHASE = 17
N_QUESTIONS = 140
N_CASES = 420
# Significance level
ALPHA = 0.05
BOOTSTRAP_N = 10_000
BOOTSTRAP_SEED = 42

ARCH_SA = "single_agent"
ARCH_MA = "multi_agent"
ARCH_UQ = "multi_agent_uq"
ARCHITECTURES = (ARCH_SA, ARCH_MA, ARCH_UQ)
ARCH_LABELS = {
    ARCH_SA: "Single-Agent",
    ARCH_MA: "Multi-Agent",
    ARCH_UQ: "Multi-Agent + UQ",
}

# Not official RAGAS
JUDGE_METRIC_LABEL = "LLM-as-judge faithfulness (Qwen3-8B, custom/RAGAS-inspired)"
# Locked T = 0.65
LOCKED_T = 0.65

PROCESSED_REL = "results/processed/phase16_cases.jsonl"
JUDGE_REL = (
    "results/raw/phase16_judge/"
    "phase16_judge_20260828T152623Z_06661255/judge.jsonl"
)
PHASE15_REL = "results/raw/phase15_benchmark/phase15_20260826T203744Z_dae9c3a4/cases.jsonl"
FROZEN_140_REL = "data/final/selected_140_questions.csv"
CAL_40_REL = "data/calibration/calibration_questions.csv"
LOCK_REL = "results/config/threshold.lock.json"

EXPECTED_PHASE15_SHA256 = "f5256ae40fa8db0d6172ff9f4083bbde6c1c4fdb47916baa73529bc8215caafa"
EXPECTED_PROCESSED_SHA256 = "e9e4f80dafffa0d0db970fb4426c9c0b81310717405389b2b7fd5ddb5b231e91"
EXPECTED_JUDGE_SHA256 = "093c4699b68e9653125fcd08e3b25b0d10a3357be3a20bc817a5a71e8498ebe3"
EXPECTED_FROZEN140_SHA256 = "88899ae9c66fb47bf4aa50e197d91aa171adcb882ae36f066bf614ff40fba087"
EXPECTED_CAL40_SHA256 = "1325b595ae1f9404802879ad06f52be7f7b63d805ab1edf5b66c3b608a478845"
EXPECTED_LOCK_SHA256 = "8981233604e64959386292d3d1fbdeebd4e983c0520fde3b4467772281687d88"

FORBIDDEN_IMPORT_MODULES = {
    "llama_cpp",
    "src.run.benchmark",
    "src.rag.single_agent",
    "src.rag.multi_agent",
    "src.rag.multi_agent_uq",
    "src.models.factory",
    "src.evaluation.judge_runner",
}


## `tests.py`

**File:** `V2/src/statistics/tests.py`

**What this file does**  
The actual statistical tests.

| Function | Plain English |
|---|---|
| `mcnemar_exact()` | Compares the paired correctness decisions between two systems. |
| `spearman_corr()` | Measures the relationship between confidence and judge faithfulness. |
| `mannwhitney()` | Compares judge faithfulness for ANSWER vs ABSTAIN (two groups, not paired). |
| `wilcoxon_paired()` | Paired continuous test, used after Shapiro fails. |
| `holm_adjust()` | Applies Holm correction to control for multiple comparisons. |
| `cohen_dz()` / `effect_cohens_g` / rank-biserial | Effect sizes next to the p-values. |
| `shapiro_wilk()` | Assumption check: do the differences look normal? |

**What goes in**  
Lists of 0/1 correctness flags, or lists of scores, already aligned by question.

**What comes out**  
p-value, Holm-adjusted p later in `analyse()`, and the effect-size fields above.


In [ ]:
"""Assumption-aware tests for Phase 17. No RAG / LLM imports."""

from __future__ import annotations

import math
from typing import Any

import numpy as np
from scipy import stats

from src.statistics.constants import ALPHA, BOOTSTRAP_N, BOOTSTRAP_SEED


def _as_float_array(values: list[float] | np.ndarray) -> np.ndarray:
    return np.asarray(values, dtype=float)


def wilson_ci(k: int, n: int, alpha: float = ALPHA) -> dict[str, float | int | None]:
    """Wilson score interval for a binomial proportion."""
    if n <= 0:
        return {"n": n, "k": k, "mean": None, "ci_low": None, "ci_high": None}
    p = k / n
    z = float(stats.norm.ppf(1 - alpha / 2))
    z2 = z * z
    denom = 1.0 + z2 / n
    centre = (p + z2 / (2 * n)) / denom
    margin = z * math.sqrt((p * (1 - p) / n) + z2 / (4 * n * n)) / denom
    return {
        "n": n,
        "k": k,
        "mean": p,
        "ci_low": max(0.0, centre - margin),
        "ci_high": min(1.0, centre + margin),
    }


# Apply Holm correction
def holm_adjust(p_values: list[float]) -> list[float]:
    """Holm–Bonferroni adjusted p-values, same order as input."""
    m = len(p_values)
    if m == 0:
        return []
    order = sorted(range(m), key=lambda i: p_values[i])
    adjusted = [0.0] * m
    running = 0.0
    for rank, idx in enumerate(order):
        raw = p_values[idx]
        if math.isnan(raw):
            adjusted[idx] = float("nan")
            continue
        candidate = min(1.0, (m - rank) * raw)
        running = max(running, candidate)
        adjusted[idx] = running
    return adjusted


def mean_sd(values: list[float] | np.ndarray) -> dict[str, float | int | None]:
    arr = _as_float_array(values)
    n = int(arr.size)
    if n == 0:
        return {"n": 0, "mean": None, "sd": None}
    mean = float(arr.mean())
    sd = float(arr.std(ddof=1)) if n > 1 else None
    return {"n": n, "mean": mean, "sd": sd}


def t_ci_mean(values: list[float] | np.ndarray, alpha: float = ALPHA) -> dict[str, Any]:
    arr = _as_float_array(values)
    summary = mean_sd(arr)
    n = int(summary["n"] or 0)
    mean = summary["mean"]
    sd = summary["sd"]
    if n < 2 or mean is None or sd is None:
        return {**summary, "ci_low": None, "ci_high": None, "df": None}
    sem = sd / math.sqrt(n)
    df = n - 1
    tcrit = float(stats.t.ppf(1 - alpha / 2, df))
    return {
        **summary,
        "ci_low": mean - tcrit * sem,
        "ci_high": mean + tcrit * sem,
        "df": df,
    }


# Check if values look normal
def shapiro_wilk(values: list[float] | np.ndarray) -> dict[str, Any]:
    arr = _as_float_array(values)
    arr = arr[np.isfinite(arr)]
    n = int(arr.size)
    if n < 3:
        return {"n": n, "statistic": None, "p_value": None, "normality_ok": False, "note": "n<3"}
    if np.unique(arr).size < 2:
        return {
            "n": n,
            "statistic": None,
            "p_value": None,
            "normality_ok": False,
            "note": "all values identical",
        }
    stat, p_value = stats.shapiro(arr)
    return {
        "n": n,
        "statistic": float(stat),
        "p_value": float(p_value),
        "normality_ok": bool(p_value >= ALPHA),
        "note": None,
    }


# Paired effect size
def cohen_dz(differences: list[float] | np.ndarray) -> float | None:
    arr = _as_float_array(differences)
    arr = arr[np.isfinite(arr)]
    if arr.size < 2:
        return None
    sd = float(arr.std(ddof=1))
    if sd == 0.0:
        return 0.0 if float(arr.mean()) == 0.0 else None
    return float(arr.mean() / sd)




In [ ]:
# Compare the two systems
def mcnemar_exact(left: list[int], right: list[int]) -> dict[str, Any]:
    """Exact McNemar test on paired binary outcomes (same questions).

    Table uses left/right correctness:
    n11 both 1, n10 left 1 right 0, n01 left 0 right 1, n00 both 0.
    """
    a = np.asarray(left, dtype=int)
    b = np.asarray(right, dtype=int)
    if a.size != b.size:
        raise ValueError("McNemar requires equal-length paired series")
    n = int(a.size)
    n11 = int(np.sum((a == 1) & (b == 1)))
    n10 = int(np.sum((a == 1) & (b == 0)))
    n01 = int(np.sum((a == 0) & (b == 1)))
    n00 = int(np.sum((a == 0) & (b == 0)))
    discordant = n10 + n01
    if discordant == 0:
        exact_p = 1.0
        chi2 = 0.0
        chi2_p = 1.0
    else:
        exact_p = float(stats.binomtest(n01, n=discordant, p=0.5, alternative="two-sided").pvalue)
        chi2 = (abs(n01 - n10) - 1) ** 2 / discordant
        chi2_p = float(stats.chi2.sf(chi2, 1))
    odds_num = n01 + 0.5
    odds_den = n10 + 0.5
    odds_ratio = odds_num / odds_den
    cohen_g = (n01 / discordant - 0.5) if discordant else 0.0
    mean_left = n11 / n + n10 / n
    mean_right = n11 / n + n01 / n
    return {
        "n": n,
        "n11_both_positive": n11,
        "n10_left_only": n10,
        "n01_right_only": n01,
        "n00_both_negative": n00,
        "n_discordant": discordant,
        "mean_left": mean_left,
        "mean_right": mean_right,
        "mean_difference": mean_right - mean_left,
        "test": "McNemar exact (binomial, two-sided)",
        "statistic": n01,
        "statistic_name": "n01 (right-only positives among discordant pairs)",
        "df": None,
        "p_value": exact_p,
        "chi2_continuity": chi2,
        "chi2_p_value": chi2_p,
        "effect_odds_ratio_haldane": odds_ratio,
        "effect_cohens_g": cohen_g,
        "ci_left": wilson_ci(n11 + n10, n),
        "ci_right": wilson_ci(n11 + n01, n),
    }


# Paired test if Shapiro fails
def wilcoxon_paired(left: list[float], right: list[float]) -> dict[str, Any]:
    """Wilcoxon signed-rank on paired continuous scores; Shapiro on differences."""
    a = _as_float_array(left)
    b = _as_float_array(right)
    if a.size != b.size:
        raise ValueError("Wilcoxon requires equal-length paired series")
    diff = b - a
    n = int(diff.size)
    n_nonzero = int(np.sum(diff != 0))
    shapiro = shapiro_wilk(diff)
    t_res = stats.ttest_rel(b, a, nan_policy="omit")
    try:
        w_res = stats.wilcoxon(b, a, zero_method="wilcox", alternative="two-sided", method="auto")
        w_stat = float(w_res.statistic)
        w_p = float(w_res.pvalue)
        w_note = None
    except ValueError as exc:
        w_stat, w_p = None, 1.0 if n_nonzero == 0 else None
        w_note = str(exc)
    selected = "Wilcoxon signed-rank"
    if shapiro.get("normality_ok") and n_nonzero >= 3:
        selected = "paired t-test (Shapiro p≥0.05); Wilcoxon also reported"
    dz = cohen_dz(diff)
    z = None
    rank_biserial = None
    if n_nonzero >= 1 and w_stat is not None:
        mean_w = n_nonzero * (n_nonzero + 1) / 4.0
        var_w = n_nonzero * (n_nonzero + 1) * (2 * n_nonzero + 1) / 24.0
        if var_w > 0:
            z = (float(w_stat) - mean_w) / math.sqrt(var_w)
            rank_biserial = z / math.sqrt(n_nonzero)
    t_ci = t_ci_mean(diff)
    return {
        "n": n,
        "n_nonzero_differences": n_nonzero,
        "mean_left": float(a.mean()) if n else None,
        "mean_right": float(b.mean()) if n else None,
        "mean_difference": float(diff.mean()) if n else None,
        "sd_difference": float(diff.std(ddof=1)) if n > 1 else None,
        "ci95_diff_low": t_ci["ci_low"],
        "ci95_diff_high": t_ci["ci_high"],
        "test": "Wilcoxon signed-rank (two-sided, zeros discarded)",
        "selected_test": selected,
        "statistic": w_stat,
        "statistic_name": "Wilcoxon W",
        "df": None,
        "p_value": w_p,
        "t_statistic": float(t_res.statistic) if t_res.statistic is not None else None,
        "t_df": int(n - 1) if n > 1 else None,
        "t_p_value": float(t_res.pvalue) if t_res.pvalue is not None else None,
        "shapiro": shapiro,
        "effect_cohens_dz": dz,
        "effect_rank_biserial_approx": rank_biserial,
        "z_approx": z,
        "note": w_note,
    }




In [ ]:
# Relationship between two scores
def spearman_corr(x: list[float], y: list[float]) -> dict[str, Any]:
    a = _as_float_array(x)
    b = _as_float_array(y)
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    n = int(a.size)
    if n < 3 or np.unique(a).size < 2 or np.unique(b).size < 2:
        return {
            "n": n,
            "test": "Spearman rank correlation (two-sided)",
            "statistic": None,
            "statistic_name": "rho",
            "df": None,
            "p_value": None,
            "note": "insufficient variation or n<3",
        }
    rho, p_value = stats.spearmanr(a, b)
    return {
        "n": n,
        "test": "Spearman rank correlation (two-sided)",
        "statistic": float(rho),
        "statistic_name": "rho",
        "df": n - 2,
        "p_value": float(p_value),
        "mean_x": float(a.mean()),
        "mean_y": float(b.mean()),
        "sd_x": float(a.std(ddof=1)),
        "sd_y": float(b.std(ddof=1)),
        "note": None,
    }


# Compare ANSWER vs ABSTAIN groups
def mannwhitney(left: list[float], right: list[float], label_left: str, label_right: str) -> dict[str, Any]:
    a = _as_float_array(left)
    b = _as_float_array(right)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    n1, n2 = int(a.size), int(b.size)
    if n1 < 1 or n2 < 1:
        return {
            "n_left": n1,
            "n_right": n2,
            "test": "Mann–Whitney U (two-sided, unpaired)",
            "statistic": None,
            "p_value": None,
            "note": "empty group",
        }
    res = stats.mannwhitneyu(a, b, alternative="two-sided")
    # rank-biserial: r = 1 - 2U/(n1 n2)
    r_rb = (2.0 * float(res.statistic)) / (n1 * n2) - 1.0
    return {
        "n_left": n1,
        "n_right": n2,
        "label_left": label_left,
        "label_right": label_right,
        "mean_left": float(a.mean()),
        "mean_right": float(b.mean()),
        "sd_left": float(a.std(ddof=1)) if n1 > 1 else None,
        "sd_right": float(b.std(ddof=1)) if n2 > 1 else None,
        "test": "Mann–Whitney U (two-sided, unpaired)",
        "statistic": float(res.statistic),
        "statistic_name": "U",
        "df": None,
        "p_value": float(res.pvalue),
        "effect_rank_biserial": r_rb,
        "note": "Groups are disjoint questions (ANSWER vs ABSTAIN), not paired.",
    }


def bootstrap_mean_diff(
    left: list[float],
    right: list[float],
    n_boot: int = BOOTSTRAP_N,
    seed: int = BOOTSTRAP_SEED,
    alpha: float = ALPHA,
) -> dict[str, Any]:
    """Paired bootstrap percentile CI for mean(right-left)."""
    a = _as_float_array(left)
    b = _as_float_array(right)
    n = int(a.size)
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        diffs[i] = float((b[idx] - a[idx]).mean())
    lo, hi = np.quantile(diffs, [alpha / 2, 1 - alpha / 2])
    observed = float((b - a).mean())
    return {
        "n": n,
        "n_boot": n_boot,
        "seed": seed,
        "observed_mean_difference": observed,
        "ci_low": float(lo),
        "ci_high": float(hi),
    }


def bootstrap_selective_minus_baseline(
    uq_answered: list[int],
    uq_correct_displayed: list[int],
    baseline_correct: list[int],
    n_boot: int = BOOTSTRAP_N,
    seed: int = BOOTSTRAP_SEED,
    alpha: float = ALPHA,
) -> dict[str, Any]:
    """Bootstrap CI for UQ selective accuracy minus baseline accuracy (question resampling)."""
    ans = np.asarray(uq_answered, dtype=int)
    corr = np.asarray(uq_correct_displayed, dtype=int)
    base = np.asarray(baseline_correct, dtype=int)
    n = int(ans.size)
    rng = np.random.default_rng(seed)
    deltas = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        n_ans = int(ans[idx].sum())
        if n_ans == 0:
            continue
        sel = float(corr[idx][ans[idx] == 1].sum() / n_ans)
        acc = float(base[idx].mean())
        deltas.append(sel - acc)
    arr = np.asarray(deltas, dtype=float)
    n_ans_obs = int(ans.sum())
    sel_obs = float(corr[ans == 1].sum() / n_ans_obs) if n_ans_obs else None
    acc_obs = float(base.mean())
    lo, hi = np.quantile(arr, [alpha / 2, 1 - alpha / 2]) if arr.size else (None, None)
    return {
        "n": n,
        "n_boot": int(arr.size),
        "seed": seed,
        "selective_accuracy": sel_obs,
        "baseline_accuracy": acc_obs,
        "observed_difference": (sel_obs - acc_obs) if sel_obs is not None else None,
        "ci_low": float(lo) if lo is not None else None,
        "ci_high": float(hi) if hi is not None else None,
        "note": (
            "Question-level bootstrap. Selective accuracy uses the ANSWER subset; "
            "this is not a paired McNemar on all 140 questions."
        ),
    }




In [ ]:
def significant(p_adjusted: float | None, alpha: float = ALPHA) -> bool:
    if p_adjusted is None or (isinstance(p_adjusted, float) and math.isnan(p_adjusted)):
        return False
    return bool(p_adjusted < alpha)


## `analysis.py`

**File:** `V2/src/statistics/analysis.py`

**What this file does**  
`analyse()` calls the tests for each research question.

- RQ1 confirmatory: `mcnemar_exact(sa_disp, ma_disp)` on displayed numeric correctness.
- RQ2 confirmatory: Spearman confidence vs judge score; Mann–Whitney ANSWER vs ABSTAIN; Wilcoxon Multi-Agent vs UQ.
- RQ3 confirmatory: McNemar on `unsupported_emitted`.
- Then `_annotate_family()` runs Holm inside each family.

**What I can say in the viva**  
If they ask where RQ1 is tested, open `analyse()` and point at `rq1_mcnemar_displayed_sa_vs_ma`.


In [ ]:
"""Phase 17 RQ1–RQ3 analysis on frozen Phase 16 outputs."""

from __future__ import annotations

from typing import Any

from src.statistics.constants import (
    ALPHA,
    ARCH_LABELS,
    ARCH_MA,
    ARCH_SA,
    ARCH_UQ,
    ARCHITECTURES,
    JUDGE_METRIC_LABEL,
    LOCKED_T,
    N_QUESTIONS,
)
from src.statistics.load import load_joined, series
from src.statistics.tests import (
    bootstrap_mean_diff,
    bootstrap_selective_minus_baseline,
    holm_adjust,
    mannwhitney,
    mcnemar_exact,
    mean_sd,
    shapiro_wilk,
    significant,
    spearman_corr,
    t_ci_mean,
    wilcoxon_paired,
    wilson_ci,
)


def _i(values: list[Any]) -> list[int]:
    return [int(v) for v in values]


def _f(values: list[Any]) -> list[float]:
    return [float(v) for v in values]


def _f_optional(values: list[Any]) -> list[float]:
    return [float(v) for v in values if v is not None]


def _annotate_family(rows: list[dict[str, Any]], family: str) -> list[dict[str, Any]]:
    pvals = [row["p_value"] if row.get("p_value") is not None else float("nan") for row in rows]
    # Apply Holm correction
    adjusted = holm_adjust(pvals)
    out = []
    for row, padj in zip(rows, adjusted):
        copied = dict(row)
        copied["family"] = family
        copied["p_value_holm"] = padj
        copied["significant_holm_0.05"] = significant(padj)
        copied["alpha"] = ALPHA
        out.append(copied)
    return out


def _arch_descriptive(joined: dict[str, Any]) -> list[dict[str, Any]]:
    rows = []
    for arch in ARCHITECTURES:
        displayed = _i(series(joined, arch, "answer_correctness"))
        claim = _i(series(joined, arch, "answer_correctness_claim"))
        answered = _i(series(joined, arch, "answered"))
        unsupported = _i(series(joined, arch, "unsupported_emitted"))
        llm = _f(series(joined, arch, "llm_faithfulness"))
        overlap = _f(series(joined, arch, "faithfulness"))
        ctx_p = _f(series(joined, arch, "context_precision"))
        ctx_r = _f(series(joined, arch, "context_recall"))
        n_answer = sum(answered)
        n_abstain = N_QUESTIONS - n_answer
        n_correct_answered = sum(d for d, a in zip(displayed, answered) if a)
        sel = wilson_ci(n_correct_answered, n_answer) if n_answer else {
            "n": 0, "k": 0, "mean": None, "ci_low": None, "ci_high": None
        }
        llm_answer = [s for s, a in zip(llm, answered) if a]
        llm_abstain = [s for s, a in zip(llm, answered) if not a]
        conf = series(joined, arch, "confidence")
        rows.append({
            "architecture": arch,
            "label": ARCH_LABELS[arch],
            "n": N_QUESTIONS,
            "n_answer": n_answer,
            "n_abstain": n_abstain,
            "coverage": n_answer / N_QUESTIONS,
            "coverage_wilson": wilson_ci(n_answer, N_QUESTIONS),
            "abstention_rate": n_abstain / N_QUESTIONS,
            "abstention_wilson": wilson_ci(n_abstain, N_QUESTIONS),
            "displayed_correct_k": sum(displayed),
            "displayed_correctness": wilson_ci(sum(displayed), N_QUESTIONS),
            "claim_correct_k": sum(claim),
            "claim_correctness": wilson_ci(sum(claim), N_QUESTIONS),
            "selective_accuracy_k": n_correct_answered,
            "selective_accuracy_n": n_answer,
            "selective_accuracy": sel,
            "unsupported_emitted_k": sum(unsupported),
            "unsupported_emitted": wilson_ci(sum(unsupported), N_QUESTIONS),
            "llm_faithfulness_all": {**mean_sd(llm), **{k: t_ci_mean(llm)[k] for k in ("ci_low", "ci_high", "df")}},
            "llm_faithfulness_answered": mean_sd(llm_answer) if llm_answer else {"n": 0, "mean": None, "sd": None},
            "llm_faithfulness_abstained": mean_sd(llm_abstain) if llm_abstain else {"n": 0, "mean": None, "sd": None},
            "token_overlap": {**mean_sd(overlap), **{k: t_ci_mean(overlap)[k] for k in ("ci_low", "ci_high", "df")}},
            "context_precision": mean_sd(ctx_p),
            "context_recall": mean_sd(ctx_r),
            "confidence": mean_sd(_f_optional(conf)),
        })
    return rows




In [ ]:
def analyse(root=None) -> dict[str, Any]:
    joined = load_joined(root)
    descriptive = _arch_descriptive(joined)

    sa_disp = _i(series(joined, ARCH_SA, "answer_correctness"))
    ma_disp = _i(series(joined, ARCH_MA, "answer_correctness"))
    uq_disp = _i(series(joined, ARCH_UQ, "answer_correctness"))
    sa_claim = _i(series(joined, ARCH_SA, "answer_correctness_claim"))
    ma_claim = _i(series(joined, ARCH_MA, "answer_correctness_claim"))
    uq_claim = _i(series(joined, ARCH_UQ, "answer_correctness_claim"))
    sa_unsup = _i(series(joined, ARCH_SA, "unsupported_emitted"))
    ma_unsup = _i(series(joined, ARCH_MA, "unsupported_emitted"))
    uq_unsup = _i(series(joined, ARCH_UQ, "unsupported_emitted"))
    sa_llm = _f(series(joined, ARCH_SA, "llm_faithfulness"))
    ma_llm = _f(series(joined, ARCH_MA, "llm_faithfulness"))
    uq_llm = _f(series(joined, ARCH_UQ, "llm_faithfulness"))
    sa_ov = _f(series(joined, ARCH_SA, "faithfulness"))
    ma_ov = _f(series(joined, ARCH_MA, "faithfulness"))
    uq_ov = _f(series(joined, ARCH_UQ, "faithfulness"))
    uq_ans = _i(series(joined, ARCH_UQ, "answered"))
    uq_conf = _f(series(joined, ARCH_UQ, "confidence"))

    # RQ1 confirmatory: SA vs MA displayed numeric correctness
    # RQ1: compare correctness
    rq1_primary = dict(mcnemar_exact(sa_disp, ma_disp))
    rq1_primary.update({
        "id": "rq1_mcnemar_displayed_sa_vs_ma",
        "rq": "RQ1",
        "role": "confirmatory",
        "left": ARCH_SA,
        "right": ARCH_MA,
        "outcome": "displayed numeric answer correctness",
        "layer": "numeric FinQA correctness (primary RQ1)",
        "unit": "frozen FinQA test question (n=140), paired across architectures",
    })
    rq1_conf = _annotate_family([rq1_primary], "rq1_confirmatory")

    rq1_expl_raw = [
        {
            **mcnemar_exact(sa_disp, uq_disp),
            "id": "rq1_mcnemar_displayed_sa_vs_uq",
            "rq": "RQ1",
            "role": "exploratory",
            "left": ARCH_SA,
            "right": ARCH_UQ,
            "outcome": "displayed numeric answer correctness",
            "layer": "numeric FinQA correctness",
            "note": "UQ ABSTAIN displayed text is the abstention template and is usually numerically incorrect.",
            "unit": "frozen FinQA test question (n=140), paired",
        },
        {
            **mcnemar_exact(ma_disp, uq_disp),
            "id": "rq1_mcnemar_displayed_ma_vs_uq",
            "rq": "RQ1",
            "role": "exploratory",
            "left": ARCH_MA,
            "right": ARCH_UQ,
            "outcome": "displayed numeric answer correctness",
            "layer": "numeric FinQA correctness",
            "unit": "frozen FinQA test question (n=140), paired",
        },
        {
            **mcnemar_exact(sa_claim, ma_claim),
            "id": "rq1_mcnemar_claim_sa_vs_ma",
            "rq": "RQ1",
            "role": "exploratory",
            "left": ARCH_SA,
            "right": ARCH_MA,
            "outcome": "claim numeric correctness (UQ uses draft)",
            "layer": "numeric FinQA correctness (claim)",
            "unit": "frozen FinQA test question (n=140), paired",
        },
        {
            **mcnemar_exact(sa_claim, uq_claim),
            "id": "rq1_mcnemar_claim_sa_vs_uq",
            "rq": "RQ1",
            "role": "exploratory",
            "left": ARCH_SA,
            "right": ARCH_UQ,
            "outcome": "claim numeric correctness (UQ uses draft)",
            "layer": "numeric FinQA correctness (claim)",
            "unit": "frozen FinQA test question (n=140), paired",
        },
        {
            **mcnemar_exact(ma_claim, uq_claim),
            "id": "rq1_mcnemar_claim_ma_vs_uq",
            "rq": "RQ1",
            "role": "exploratory",
            "left": ARCH_MA,
            "right": ARCH_UQ,
            "outcome": "claim numeric correctness (UQ uses draft)",
            "layer": "numeric FinQA correctness (claim)",
            "unit": "frozen FinQA test question (n=140), paired",
        },
    ]
    rq1_expl = _annotate_family(rq1_expl_raw, "rq1_exploratory")

    # RQ2 confirmatory
    llm_ans = [s for s, a in zip(uq_llm, uq_ans) if a]
    llm_abs = [s for s, a in zip(uq_llm, uq_ans) if not a]
    rq2_conf_raw = [
        {
            # RQ2: confidence vs judge faithfulness
            **spearman_corr(uq_conf, uq_llm),
            "id": "rq2_spearman_uq_confidence_vs_llm_faithfulness",
            "rq": "RQ2",
            "role": "confirmatory",
            "outcome": "UQ confidence vs LLM-as-judge faithfulness (all 140, draft claim)",
            "layer": JUDGE_METRIC_LABEL,
            "unit": "frozen FinQA test question (n=140) within multi_agent_uq",
        },
        {
            # RQ2: ANSWER vs ABSTAIN
            **mannwhitney(llm_ans, llm_abs, "UQ ANSWER", "UQ ABSTAIN"),
            "id": "rq2_mannwhitney_uq_llm_answer_vs_abstain",
            "rq": "RQ2",
            "role": "confirmatory",
            "outcome": "LLM-as-judge faithfulness, UQ ANSWER vs ABSTAIN",
            "layer": JUDGE_METRIC_LABEL,
            "unit": "question grouped by UQ decision (unpaired; n_ANSWER=78, n_ABSTAIN=62)",
        },
        {
            **wilcoxon_paired(ma_llm, uq_llm),
            "id": "rq2_wilcoxon_llm_ma_vs_uq",
            "rq": "RQ2",
            "role": "confirmatory",
            "left": ARCH_MA,
            "right": ARCH_UQ,
            "outcome": "LLM-as-judge faithfulness (all 140; UQ includes abstained drafts)",
            "layer": JUDGE_METRIC_LABEL,
            "unit": "frozen FinQA test question (n=140), paired",
        },
    ]
    rq2_conf = _annotate_family(rq2_conf_raw, "rq2_confirmatory")

    conf_ans = [c for c, a in zip(uq_conf, uq_ans) if a]
    llm_ans_only = [s for s, a in zip(uq_llm, uq_ans) if a]
    rq2_expl_raw = [
        {
            **wilcoxon_paired(sa_llm, ma_llm),
            "id": "rq2_wilcoxon_llm_sa_vs_ma",
            "rq": "RQ2",
            "role": "exploratory",
            "left": ARCH_SA,
            "right": ARCH_MA,


In [ ]:
            "outcome": "LLM-as-judge faithfulness",
            "layer": JUDGE_METRIC_LABEL,
            "unit": "frozen FinQA test question (n=140), paired",
        },
        {
            **wilcoxon_paired(sa_llm, uq_llm),
            "id": "rq2_wilcoxon_llm_sa_vs_uq",
            "rq": "RQ2",
            "role": "exploratory",
            "left": ARCH_SA,
            "right": ARCH_UQ,
            "outcome": "LLM-as-judge faithfulness",
            "layer": JUDGE_METRIC_LABEL,
            "unit": "frozen FinQA test question (n=140), paired",
        },
        {
            **spearman_corr(uq_conf, _f(uq_claim)),
            "id": "rq2_spearman_uq_confidence_vs_claim_correctness",
            "rq": "RQ2",
            "role": "exploratory",
            "outcome": "UQ confidence vs numeric claim correctness",
            "layer": "numeric FinQA correctness (claim) vs UQ confidence",
            "unit": "frozen FinQA test question (n=140) within multi_agent_uq",
        },
        {
            **spearman_corr(conf_ans, llm_ans_only),
            "id": "rq2_spearman_uq_confidence_vs_llm_among_answered",
            "rq": "RQ2",
            "role": "exploratory",
            "outcome": "UQ confidence vs LLM-as-judge faithfulness (ANSWER only)",
            "layer": JUDGE_METRIC_LABEL,
            "unit": "UQ ANSWER questions only (n=78)",
        },
    ]
    rq2_expl = _annotate_family(rq2_expl_raw, "rq2_exploratory")

    # RQ3
    rq3_conf_raw = [
        {
            # RQ3: compare unsupported-emitted
            **mcnemar_exact(sa_unsup, uq_unsup),
            "id": "rq3_mcnemar_unsupported_sa_vs_uq",
            "rq": "RQ3",
            "role": "confirmatory",
            "left": ARCH_SA,
            "right": ARCH_UQ,
            "outcome": "unsupported_emitted (ANSWER and displayed numeric incorrect)",
            "layer": "coverage / selective accuracy / abstention (RQ3); not a hallucination label",
            "unit": "frozen FinQA test question (n=140), paired",
        },
        {
            **mcnemar_exact(ma_unsup, uq_unsup),
            "id": "rq3_mcnemar_unsupported_ma_vs_uq",
            "rq": "RQ3",
            "role": "confirmatory",
            "left": ARCH_MA,
            "right": ARCH_UQ,
            "outcome": "unsupported_emitted (ANSWER and displayed numeric incorrect)",
            "layer": "coverage / selective accuracy / abstention (RQ3); not a hallucination label",
            "unit": "frozen FinQA test question (n=140), paired",
        },
    ]
    rq3_conf = _annotate_family(rq3_conf_raw, "rq3_confirmatory")

    n_true_abstain = sum(1 for a, c in zip(uq_ans, uq_claim) if (not a) and (not c))
    n_false_abstain = sum(1 for a, c in zip(uq_ans, uq_claim) if (not a) and c)
    n_true_answer = sum(1 for a, d in zip(uq_ans, uq_disp) if a and d)
    n_false_answer = sum(1 for a, d in zip(uq_ans, uq_disp) if a and not d)

    sel_vs_sa = bootstrap_selective_minus_baseline(uq_ans, uq_disp, sa_disp)
    sel_vs_ma = bootstrap_selective_minus_baseline(uq_ans, uq_disp, ma_disp)

    secondary_overlap = _annotate_family(
        [
            {**wilcoxon_paired(sa_ov, ma_ov), "id": "sec_overlap_sa_vs_ma", "left": ARCH_SA, "right": ARCH_MA},
            {**wilcoxon_paired(sa_ov, uq_ov), "id": "sec_overlap_sa_vs_uq", "left": ARCH_SA, "right": ARCH_UQ},
            {**wilcoxon_paired(ma_ov, uq_ov), "id": "sec_overlap_ma_vs_uq", "left": ARCH_MA, "right": ARCH_UQ},
        ],
        "secondary_token_overlap",
    )
    for row in secondary_overlap:
        row["rq"] = "secondary"
        row["role"] = "secondary"
        row["outcome"] = "CPU token-overlap faithfulness"
        row["layer"] = "token-overlap (secondary; not official RAGAS; not RQ2 primary)"
        row["unit"] = "frozen FinQA test question (n=140), paired"

    ctx_p_sa = _f(series(joined, ARCH_SA, "context_precision"))
    ctx_p_ma = _f(series(joined, ARCH_MA, "context_precision"))
    ctx_p_uq = _f(series(joined, ARCH_UQ, "context_precision"))
    retrieval_note = (
        "Context precision and recall are identical across architectures by design "
        "(shared retrieval). They are retrieval-control metrics, not architecture tests."
    )
    ctx_identical = (
        ctx_p_sa == ctx_p_ma == ctx_p_uq
        and _f(series(joined, ARCH_SA, "context_recall"))
        == _f(series(joined, ARCH_MA, "context_recall"))
        == _f(series(joined, ARCH_UQ, "context_recall"))
    )

    # Assumption checks (Shapiro)
    assumptions = []
    for name, left, right in (
        ("llm_sa_vs_ma", sa_llm, ma_llm),
        ("llm_ma_vs_uq", ma_llm, uq_llm),
        ("llm_sa_vs_uq", sa_llm, uq_llm),
        ("overlap_sa_vs_ma", sa_ov, ma_ov),
        ("uq_confidence", uq_conf, uq_conf),
    ):
        if name == "uq_confidence":
            assumptions.append({"name": name, "shapiro": shapiro_wilk(uq_conf), "metric": "UQ confidence"})
        else:
            diff = [r - l for l, r in zip(left, right)]
            assumptions.append({"name": name, "shapiro": shapiro_wilk(diff), "metric": name})

    interpretation = {
        "statistical_unit": (
            "The statistical unit is the frozen FinQA test question (n=140). "
            "The same 140 questions were evaluated independently on three architectures "
            "(no RAG1→RAG2→RAG3 chaining). Between-architecture tests are paired on question_id. "
            "Do not treat 420 cases as independent samples for architecture comparisons."
        ),
        "layers": {
            "rq1": "numeric FinQA displayed answer correctness (rel_tol=0.01) is the primary RQ1 measure",
            "rq2": JUDGE_METRIC_LABEL + " — not official RAGAS",
            "rq3": "coverage, selective accuracy, abstention, unsupported_emitted at locked T=0.65",
            "retrieval": "context precision/recall (gold file_name / context_id)",
            "secondary": "CPU token-overlap faithfulness",
        },
        "rq1": None,
        "rq2": None,
        "rq3": None,
        "limitations": [
            "Same-model Qwen3-8B judge; not official RAGAS Faithfulness.",
            "unsupported_emitted is answered-and-numerically-wrong, not a labelled hallucination corpus.",
            "Selective accuracy uses a selected subset (UQ ANSWER); it is not a paired accuracy on all 140.",
            "T=0.65 was locked on DEV 40 only; it was not retuned on the frozen 140.",
            "Questions are a frozen sample of FinQA test; company repeats may induce weak dependence.",
            "Phase 17 does not rerun RAG, Qwen generation, or the judge.",
        ],
    }

    # Fill interpretation after seeing Holm results (honest)
    p_rq1 = rq1_conf[0]["p_value_holm"]


In [ ]:
    rho_conf = rq2_conf[0].get("statistic")
    p_rho = rq2_conf[0]["p_value_holm"]
    p_mw = rq2_conf[1]["p_value_holm"]
    p_w = rq2_conf[2]["p_value_holm"]
    p_unsup_sa = rq3_conf[0]["p_value_holm"]
    p_unsup_ma = rq3_conf[1]["p_value_holm"]
    interpretation["rq1"] = (
        f"Confirmatory McNemar (exact binomial) on displayed FinQA numeric correctness, "
        f"Single-Agent vs Multi-Agent, n=140 paired questions: SA {sum(sa_disp)}/140 "
        f"(Wilson 95% CI {wilson_ci(sum(sa_disp), 140)['ci_low']:.4f}–{wilson_ci(sum(sa_disp), 140)['ci_high']:.4f}) "
        f"vs MA {sum(ma_disp)}/140 "
        f"(Wilson 95% CI {wilson_ci(sum(ma_disp), 140)['ci_low']:.4f}–{wilson_ci(sum(ma_disp), 140)['ci_high']:.4f}); "
        f"discordant pairs {rq1_conf[0]['n_discordant']} (SA-only {rq1_conf[0]['n10_left_only']}, "
        f"MA-only {rq1_conf[0]['n01_right_only']}); exact p={rq1_conf[0]['p_value']:.4g}; "
        f"Holm-adjusted p={p_rq1:.4g} (family size 1); Cohen's g={rq1_conf[0]['effect_cohens_g']:.4f}. "
        "This is not statistically significant at α=0.05. "
        "The data do not support a Multi-Agent accuracy improvement over Single-Agent."
    )
    interpretation["rq2"] = (
        f"UQ confidence is positively associated with {JUDGE_METRIC_LABEL} "
        f"(Spearman ρ={rho_conf:.4f}, df=138, p={rq2_conf[0]['p_value']:.4g}, "
        f"Holm p={p_rho:.4g}). "
        f"ANSWER cases have higher judge faithfulness than ABSTAIN cases "
        f"(means {sum(llm_ans)/len(llm_ans):.4f} vs {sum(llm_abs)/len(llm_abs):.4f}; "
        f"Mann–Whitney U={rq2_conf[1]['statistic']:.1f}, p={rq2_conf[1]['p_value']:.4g}, "
        f"Holm p={p_mw:.4g}). "
        f"Paired Wilcoxon of the same judge score, Multi-Agent vs UQ on all 140 questions, "
        f"is not significant (W={rq2_conf[2]['statistic']}, p={rq2_conf[2]['p_value']:.4g}, "
        f"Holm p={p_w:.4g}). "
        "Holm family size=3. "
        "Confidence therefore tracks support/abstention within UQ, but UQ does not significantly "
        "raise mean faithfulness versus always-answer Multi-Agent on the full paired set "
        "(abstained drafts pull the UQ mean down). Not official RAGAS."
    )
    interpretation["rq3"] = (
        f"At locked T=0.65, UQ coverage is {sum(uq_ans)}/140="
        f"{sum(uq_ans)/N_QUESTIONS:.4f} (Wilson 95% CI "
        f"{wilson_ci(sum(uq_ans), 140)['ci_low']:.4f}–{wilson_ci(sum(uq_ans), 140)['ci_high']:.4f}); "
        f"selective displayed accuracy {n_true_answer}/{sum(uq_ans)}="
        f"{n_true_answer/sum(uq_ans):.4f} (Wilson 95% CI "
        f"{wilson_ci(n_true_answer, sum(uq_ans))['ci_low']:.4f}–"
        f"{wilson_ci(n_true_answer, sum(uq_ans))['ci_high']:.4f}). "
        f"Abstention outcomes on the draft: true abstain (incorrect draft) {n_true_abstain}; "
        f"false abstain (correct draft withheld) {n_false_abstain}. "
        f"Unsupported-emitted rate falls from SA {sum(sa_unsup)}/140 and MA {sum(ma_unsup)}/140 "
        f"to UQ {sum(uq_unsup)}/140; confirmatory McNemar Holm p="
        f"{p_unsup_sa:.4g} vs SA and {p_unsup_ma:.4g} vs MA (both significant at α=0.05). "
        f"Bootstrap 95% CI for (UQ selective accuracy − SA accuracy) is "
        f"{sel_vs_sa['ci_low']:.4f} to {sel_vs_sa['ci_high']:.4f} "
        f"(observed {sel_vs_sa['observed_difference']:.4f}). "
        "Abstention therefore reduces emitted numeric errors at the cost of coverage; "
        "this is not a labelled hallucination corpus."
    )

    return {
        "phase": 17,
        "n_questions": N_QUESTIONS,
        "n_cases": 420,
        "threshold": LOCKED_T,
        "alpha": ALPHA,
        "judge_metric_label": JUDGE_METRIC_LABEL,
        "hashes": joined["hashes"],
        "source": {
            "processed": "results/processed/phase16_cases.jsonl",
            "judge": (
                "results/raw/phase16_judge/"
                "phase16_judge_20260828T152623Z_06661255/judge.jsonl"
            ),
            "phase15_sha_verified": True,
            "used_rag_rerun": False,
        },
        "descriptive": descriptive,
        "tests": rq1_conf + rq1_expl + rq2_conf + rq2_expl + rq3_conf + secondary_overlap,
        "rq1_confirmatory": rq1_conf,
        "rq1_exploratory": rq1_expl,
        "rq2_confirmatory": rq2_conf,
        "rq2_exploratory": rq2_expl,
        "rq3_confirmatory": rq3_conf,
        "secondary_token_overlap": secondary_overlap,
        "rq3_abstention_outcomes": {
            "n_answer": sum(uq_ans),
            "n_abstain": N_QUESTIONS - sum(uq_ans),
            "true_positive_answer_displayed_correct": n_true_answer,
            "false_positive_answer_displayed_incorrect": n_false_answer,
            "true_abstain_incorrect_draft": n_true_abstain,
            "false_abstain_correct_draft": n_false_abstain,
            "threshold": LOCKED_T,
        },
        "rq3_bootstrap": {
            "selective_minus_single_agent": sel_vs_sa,
            "selective_minus_multi_agent": sel_vs_ma,
        },
        "rq2_bootstrap_llm_ma_vs_uq": bootstrap_mean_diff(ma_llm, uq_llm),
        "assumptions": assumptions,
        "retrieval_control": {
            "context_precision_identical": ctx_identical,
            "note": retrieval_note,
            "mean_precision": descriptive[0]["context_precision"]["mean"],
            "mean_recall": descriptive[0]["context_recall"]["mean"],
        },
        "interpretation": interpretation,
        "series": {
            "question_ids": joined["question_ids"],
            "sa_disp": sa_disp,
            "ma_disp": ma_disp,
            "uq_disp": uq_disp,
            "uq_ans": uq_ans,
            "uq_conf": uq_conf,
            "sa_llm": sa_llm,
            "ma_llm": ma_llm,
            "uq_llm": uq_llm,
        },
    }


## `load.py`

**File:** `V2/src/statistics/load.py`

**What this file does**  
Joins the frozen Phase 16 CPU rows to the official judge JSONL. It checks SHA-256 hashes and refuses if T is not 0.65.

**What goes in**  
Saved metric rows + judge rows.

**What comes out**  
One record per question per architecture, including `llm_faithfulness`.

**What I can say in the viva**  
Statistics never call `run_single_agent` or the judge again. `FORBIDDEN_IMPORT_MODULES` blocks that.


In [ ]:
"""Load frozen Phase 16 scored cases + official judge JSONL. Read-only SHA gates."""

from __future__ import annotations

import ast
import hashlib
import json
from pathlib import Path
from typing import Any

from src.calibration.lock import EXPECTED_LOCKED_THRESHOLD, load_official_lock
from src.config import project_root
from src.statistics.constants import (
    ARCHITECTURES,
    CAL_40_REL,
    EXPECTED_CAL40_SHA256,
    EXPECTED_FROZEN140_SHA256,
    EXPECTED_JUDGE_SHA256,
    EXPECTED_LOCK_SHA256,
    EXPECTED_PHASE15_SHA256,
    EXPECTED_PROCESSED_SHA256,
    FORBIDDEN_IMPORT_MODULES,
    FROZEN_140_REL,
    JUDGE_REL,
    LOCKED_T,
    LOCK_REL,
    N_CASES,
    N_QUESTIONS,
    PHASE15_REL,
    PROCESSED_REL,
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            text = line.strip()
            if text:
                rows.append(json.loads(text))
    return rows


def verify_no_generation_stack() -> None:
    stats_dir = Path(__file__).resolve().parent
    imported: set[str] = set()
    for py in stats_dir.glob("*.py"):
        tree = ast.parse(py.read_text(encoding="utf-8"))
        for node in ast.walk(tree):
            if isinstance(node, ast.Import):
                imported.update(alias.name for alias in node.names)
            elif isinstance(node, ast.ImportFrom) and node.module:
                imported.add(node.module)
    banned = imported & FORBIDDEN_IMPORT_MODULES
    if banned:
        raise RuntimeError(f"Phase 17 statistics must not import generation stack: {sorted(banned)}")


def verify_frozen_hashes(root: Path | None = None) -> dict[str, str]:
    root = root or project_root()
    lock = load_official_lock()
    if abs(float(lock["threshold"]) - EXPECTED_LOCKED_THRESHOLD) > 1e-9:
        raise RuntimeError("Locked T is not 0.65. Phase 17 must not recalibrate.")
    if abs(float(lock["threshold"]) - LOCKED_T) > 1e-9:
        raise RuntimeError("Unexpected lock threshold.")
    paths = {
        "phase15": root / PHASE15_REL,
        "processed": root / PROCESSED_REL,
        "judge": root / JUDGE_REL,
        "frozen140": root / FROZEN_140_REL,
        "cal40": root / CAL_40_REL,
        "lock": root / LOCK_REL,
    }
    expected = {
        "phase15": EXPECTED_PHASE15_SHA256,
        "processed": EXPECTED_PROCESSED_SHA256,
        "judge": EXPECTED_JUDGE_SHA256,
        "frozen140": EXPECTED_FROZEN140_SHA256,
        "cal40": EXPECTED_CAL40_SHA256,
        "lock": EXPECTED_LOCK_SHA256,
    }
    observed: dict[str, str] = {}
    for key, path in paths.items():
        if not path.is_file():
            raise FileNotFoundError(f"Required Phase 17 input missing: {path}")
        digest = sha256_file(path)
        observed[key] = digest
        if digest != expected[key]:
            raise RuntimeError(
                f"SHA-256 mismatch for {key}: {digest} != {expected[key]}. "
                "Phase 17 refuses to proceed if frozen artefacts changed."
            )
    return observed




In [ ]:
def load_joined(root: Path | None = None) -> dict[str, Any]:
    """Join Phase 16 CPU rows with official judge scores. Does not rewrite inputs."""
    verify_no_generation_stack()
    root = root or project_root()
    # Check frozen files were not changed
    hashes = verify_frozen_hashes(root)
    processed = _jsonl(root / PROCESSED_REL)
    judge = _jsonl(root / JUDGE_REL)
    if len(processed) != N_CASES or len(judge) != N_CASES:
        raise ValueError(f"Expected {N_CASES} processed and judge rows")

    judge_by_key = {str(row["case_key"]): row for row in judge}
    if len(judge_by_key) != N_CASES:
        raise ValueError("Duplicate or missing judge case_key")

    by_q: dict[str, dict[str, dict[str, Any]]] = {}
    for row in processed:
        key = str(row["case_key"])
        qid = str(row["question_id"])
        arch = str(row["architecture"])
        if key not in judge_by_key:
            raise KeyError(f"Judge missing {key}")
        jrow = judge_by_key[key]
        merged = dict(row)
        # Join the judge score
        merged["llm_faithfulness"] = float(jrow["parsed_faithfulness_score"])
        merged["llm_parse_failure"] = bool(jrow.get("parse_failure"))
        merged["judge_claim_source"] = jrow.get("claim_source")
        merged["judge_used_rag_rerun"] = bool(jrow.get("used_rag_rerun"))
        merged["judge_used_gold_context"] = bool(jrow.get("used_gold_context"))
        merged["judge_used_gold_answer"] = bool(jrow.get("used_gold_answer"))
        by_q.setdefault(qid, {})[arch] = merged

    if len(by_q) != N_QUESTIONS:
        raise ValueError(f"Expected {N_QUESTIONS} questions, found {len(by_q)}")
    for qid, arches in by_q.items():
        missing = [a for a in ARCHITECTURES if a not in arches]
        if missing:
            raise ValueError(f"{qid} missing {missing}")

    question_ids = sorted(by_q)
    return {
        "question_ids": question_ids,
        "by_question": by_q,
        "hashes": hashes,
        "n_questions": N_QUESTIONS,
        "n_cases": N_CASES,
        "used_rag_rerun": False,
        "used_llm_inference": False,
        "modifies_phase15_raw": False,
        "modifies_phase16_cpu": False,
        "modifies_phase16_judge": False,
        "threshold": LOCKED_T,
    }


def series(joined: dict[str, Any], architecture: str, field: str) -> list[Any]:
    out: list[Any] = []
    for qid in joined["question_ids"]:
        out.append(joined["by_question"][qid][architecture][field])
    return out


# 7. Streamlit


## `streamlit_app.py`

**File:** `V2/app/streamlit_app.py`

**What this file does**  
The live artefact. It runs the three real pipelines on a typed question, and also has read-only pages for frozen results and frozen questions.

**What goes in**  
A fresh question or a frozen TEST question, the existing Chroma index, and the Qwen backend on Colab.

**What comes out**  
The three pages: Live RAG Demo, Benchmark Results, Benchmark Questions. The live page shows evidence, answer, verification, confidence, T, and ANSWER/ABSTAIN.

**What I can say in the viva**  
This app does not look up Phase 15 answers. `run_live_comparison()` calls the same backend functions shown above. Launch it from the Phase 21 notebook, not by rerunning research jobs here.


In [ ]:
"""V2 live artefact: run all three RAG architectures on a fresh or frozen question.

Uses the existing Phase 6 knowledge base and Phase 8–10 pipelines.
Does not look up precomputed benchmark answers.
"""

from __future__ import annotations

import json
import os
import sys
from pathlib import Path

V2_ROOT = Path(__file__).resolve().parents[1]
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

import streamlit as st

from app.benchmark_ui import render_benchmark_questions_page, render_benchmark_results_page
from src.config import get_path, load_experiment_config, project_root
from src.rag.benchmark_catalogue import (
    apply_catalogue_prefill_to_live_input,
    apply_pending_app_page,
)
from src.models.factory import create_backend
from src.models.runtime_guard import (
    LiveRuntimeError,
    live_demo_locked,
    verify_live_llama_cpp_runtime,
)
from src.rag.live import (
    ARCHITECTURE_LABELS,
    FRESH_KB_QUESTION,
    INSUFFICIENT_EVIDENCE_QUESTION,
    INSUFFICIENT_EVIDENCE_QUESTION_ID,
    LIVE_ARCHITECTURES,
    LIVE_FAILURE_DECISIONS,
    format_confidence_display,
    format_optional,
    format_threshold_display,
    load_frozen_questions,
    resolve_displayed_confidence,
    resolve_live_locked_threshold,
    run_live_comparison,
    uq_ui_confidence_overlay,
)
from src.rag.schema import ARCHITECTURE_MULTI_AGENT_UQ, RAGCaseResult
from src.retrieval.index import COLLECTION_NAME
from src.retrieval.preflight import IndexPreflightError, validate_index_preflight
from src.utils import create_run_id


@st.cache_resource
# Load the LLM once
def _cached_backend(backend_name: str):
    if live_demo_locked():
        backend_name = "llama_cpp"
    if backend_name in {"mock", "test", "ollama", "ollama_dev"} and live_demo_locked():
        raise LiveRuntimeError("Mock/Ollama are forbidden for the Colab live demo. Use llama_cpp.")
    config = load_experiment_config()
    model_cfg = dict(config.section("model"))
    model_cfg["backend"] = backend_name
    return create_backend(model_cfg)


@st.cache_data
def _cached_frozen_questions() -> list[dict[str, str]]:
    return load_frozen_questions()


def _index_preflight() -> dict:
    config = load_experiment_config()
    retrieval_cfg = config.section("retrieval")
    index_dir = get_path(config, "kb_index")
    manifest_rel = str(retrieval_cfg.get("index_manifest") or "knowledge_base/index/index_manifest.json")
    manifest_file = (project_root() / manifest_rel).resolve()
    collection_name = str(retrieval_cfg.get("collection_name") or COLLECTION_NAME)
    return validate_index_preflight(
        index_dir,
        manifest_path=manifest_file,
        collection_name=collection_name,
    )


def _save_live_result(comparison) -> Path:
    out_dir = get_path(load_experiment_config(), "results_raw")
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / "live_sessions.jsonl"
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(comparison.to_dict()) + "\n")
    return path


# Show the retrieved chunks
def render_evidence(chunks: list[dict]) -> None:
    if not chunks:
        st.caption("No evidence retrieved.")
        return
    for idx, chunk in enumerate(chunks, start=1):
        score = chunk.get("score")
        title = f"[{idx}] {chunk.get('file_name') or chunk.get('doc_id') or 'chunk'}"
        if score is not None:
            title += f" · score {float(score):.4f}"
        with st.expander(title, expanded=idx == 1):
            st.markdown(
                f"**chunk_id:** `{chunk.get('chunk_id')}`  \n"
                f"**company:** {chunk.get('company_symbol')} · **year:** {chunk.get('report_year')}  \n"
                f"**role:** {chunk.get('role')} · **split:** {chunk.get('split')}"
            )
            st.write(chunk.get("text") or "")


# Draw one architecture result
def render_architecture(result: RAGCaseResult) -> None:
    label = ARCHITECTURE_LABELS.get(result.architecture, result.architecture)
    st.subheader(label)
    st.caption(f"`{result.architecture}` · `{result.case_key}`")

    failed = result.decision in LIVE_FAILURE_DECISIONS or bool(result.error) or not result.retrieved_evidence
    if failed:
        st.error(f"Status: {result.decision if result.decision in LIVE_FAILURE_DECISIONS else 'ERROR / UNAVAILABLE'}")
        st.error(result.error or "Retrieval or generation failed; this is not a successful RAG run.")
        st.markdown("**Generated answer**")
        st.caption("No answer (run failed). Nothing was fabricated.")
        col_a, col_b, col_c, col_d = st.columns(4)
        col_a.metric("Confidence", "n/a")
        col_b.markdown("**Threshold**")
        col_b.write(format_threshold_display(result))
        col_c.metric("Latency (s)", format_optional(result.latency_seconds))
        col_d.metric("Evidence chunks", str(len(result.retrieved_evidence or [])))
        st.markdown("**Verification**")
        st.caption("Not available — run failed.")
        st.markdown("**Runtime**")
        st.write(
            {
                "backend": result.backend,
                "model": result.model,
                "device": result.device,
                "error": result.error,
            }
        )
        st.markdown("**Retrieved evidence / scores / metadata**")


In [ ]:
        render_evidence(result.retrieved_evidence or [])
        return

    overlay = uq_ui_confidence_overlay(result)
    decision = result.decision or "n/a"
    heading = overlay["decision_heading"] if overlay["show"] and overlay["decision_heading"] else decision
    # Show ANSWER or ABSTAIN
    if decision == "ABSTAIN":
        st.error(f"Decision: {heading}")
    else:
        st.success(f"Decision: {decision}")
    if overlay["show"] and overlay["warning"]:
        st.warning(overlay["warning"])
    if overlay["show"] and overlay["note"]:
        st.caption(overlay["note"])

    st.markdown("**Generated answer**")
    st.write(result.answer)

    if result.architecture == ARCHITECTURE_MULTI_AGENT_UQ and resolve_displayed_confidence(result) is None:
        st.error("UQ confidence could not be calculated. Displaying n/a — this is not a valid 0.0 confidence.")

    col_a, col_b, col_c, col_d = st.columns(4)
    # Do not use st.metric for 0–1 scores: Streamlit can render 0.7688 / 0.55 as 0.
    col_a.markdown("**Confidence**")
    col_a.write(format_confidence_display(result))
    col_b.markdown("**Threshold**")
    col_b.write(format_threshold_display(result))
    col_c.metric("Latency (s)", format_optional(result.latency_seconds))
    col_d.metric("Evidence chunks", str(len(result.retrieved_evidence or [])))

    st.markdown("**Verification**")
    verify = result.verification_result
    if not verify:
        st.caption("Not applicable for this architecture.")
    else:
        st.write(
            {
                "status": verify.get("status"),
                "rationale": verify.get("rationale"),
                "verification_score": verify.get("verification_score"),
                "lexical_score": verify.get("lexical_score"),
                "llm_score": verify.get("llm_score"),
                "verification_threshold": verify.get("verification_threshold"),
            }
        )
        if verify.get("rationale") and not str(verify.get("rationale", "")).startswith(str(verify.get("status") or "")):
            st.error("Verification status and rationale are inconsistent.")

    uq = (result.configuration or {}).get("uncertainty_result")
    if uq:
        st.markdown("**Uncertainty**")
        st.write(
            {
                "method": uq.get("method"),
                "retrieval_score": uq.get("retrieval_score"),
                "verification_score": uq.get("verification_score"),
                "confidence": uq.get("confidence"),
                "displayed_confidence": format_confidence_display(result),
                "displayed_threshold": format_threshold_display(result),
                "decision": result.decision,
            }
        )

    st.markdown("**Runtime**")
    st.write(
        {
            "backend": result.backend,
            "model": result.model,
            "quantisation": result.quantisation,
            "device": result.device,
            "gpu": result.gpu,
            "latency_seconds": result.latency_seconds,
        }
    )

    st.markdown("**Retrieved evidence / scores / metadata**")
    if result.retrieval_scores:
        st.caption("Retrieval scores: " + ", ".join(f"{s:.4f}" for s in result.retrieval_scores))
    render_evidence(result.retrieved_evidence or [])


def render_live_rag_demo() -> None:
    st.title("V2 Live RAG Comparison")
    st.write(
        "Ask a **fresh question** or replay a **frozen FinQA test case**. "
        "Each architecture runs independently on the same original question "
        "using the shared Phase 6 knowledge base. This is not a benchmark lookup."
    )

    # Load locked T = 0.65
    locked_t = resolve_live_locked_threshold()

    with st.sidebar:
        st.header("Runtime")
        require_cuda = os.environ.get("V2_REQUIRE_CUDA", "1") == "1" or live_demo_locked()
        if live_demo_locked():
            try:
                runtime = verify_live_llama_cpp_runtime(require_cuda=True)
            except LiveRuntimeError as exc:
                st.error(str(exc))
                st.error("Do not open http://127.0.0.1:8501 on the Mac. Use the Colab notebook proxy URL.")
                st.stop()
            backend_name = "llama_cpp"
            st.success(f"Backend locked: llama_cpp · GPU: {runtime.get('gpu')} · chunks: {runtime.get('index_chunks')}")
            st.caption("Mock and Ollama are disabled. This process must be Colab CUDA, not mps_capable_host.")
        else:
            backend_options = ["auto", "ollama_dev", "llama_cpp", "mock"]
            env_backend = os.environ.get("V2_LIVE_BACKEND", "auto").strip().lower()
            if env_backend not in backend_options:
                env_backend = "auto"
            backend_name = st.selectbox(
                "LLM backend",
                options=backend_options,
                index=backend_options.index(env_backend),
                help="Colab live demo locks llama_cpp. mock/ollama_dev are local UI/testing only.",
            )
            if backend_name == "llama_cpp":
                try:
                    runtime = verify_live_llama_cpp_runtime(require_cuda=require_cuda)
                    st.success(f"llama_cpp on {runtime.get('gpu')} · device={runtime.get('device')}")
                except LiveRuntimeError as exc:
                    st.error(str(exc))
                    st.error("llama_cpp live demo requires Colab CUDA. This Mac process would report mps_capable_host.")
                    st.stop()
            if backend_name == "mock":
                st.warning("Mock backend is for UI/testing only. It must not be treated as a real RAG answer.")
            if backend_name in {"ollama", "ollama_dev"}:
                st.warning("Ollama is local-dev only. The Colab live demo must use llama_cpp.")
        st.markdown(f"**Locked threshold T = {locked_t:.2f}**")
        st.caption(
            "Official lock from `results/config/threshold.lock.json` (FinQA DEV 40 only). "
            "Not editable. Not the smoke 0.55 fallback. Not tuned on the frozen 140."
        )
        save_raw = st.checkbox("Append raw result to results/raw/live_sessions.jsonl", value=True)
        st.caption("Frozen 140 / calibration 40 / Phase 15–18 results are not modified.")

        try:
            preflight = _index_preflight()
            st.success(
                f"KB preflight PASS · {preflight['actual_count']} chunks"
            )


In [ ]:
        except IndexPreflightError as exc:
            st.error(str(exc))
            st.stop()

    source = st.radio(
        "Question source",
        options=["Fresh question", "Frozen test case", "Insufficient-evidence demo"],
        horizontal=True,
        key="question_source",
    )
    frozen = _cached_frozen_questions()
    reference_answer = None
    question_id = None
    question_source = "fresh"

    if source == "Insufficient-evidence demo":
        question = INSUFFICIENT_EVIDENCE_QUESTION
        question_id = INSUFFICIENT_EVIDENCE_QUESTION_ID
        question_source = "insufficient"
        st.warning(
            "This question is outside the FinQA source-PDF corpus. "
            "It is for demonstrating weak support / possible ABSTAIN. Abstention is not forced."
        )
        st.write(question)
    elif source == "Frozen test case":
        labels = [f"{row['id']}: {row['question'][:90]}" for row in frozen]
        chosen = st.selectbox("Frozen 140 question", options=labels)
        row = frozen[labels.index(chosen)]
        question = row["question"]
        question_id = row["id"]
        reference_answer = row.get("program_answer")
        question_source = "frozen"
        st.info(f"Using frozen question `{question_id}` (read-only).")
        st.write(question)
    else:
        if st.session_state.get("catalogue_source_id"):
            st.info(
                f"Question text copied from frozen `{st.session_state['catalogue_source_id']}`. "
                "Dataset gold was not copied. Run uses live RAG, not saved benchmark outputs."
            )
        question = st.text_area(
            "Fresh question",
            placeholder=FRESH_KB_QUESTION,
            height=120,
            key="fresh_question_text",
        )
        question_source = "fresh"
        question_id = None
        reference_answer = None

    if st.button("Run all three architectures", type="primary"):
        if not (question or "").strip():
            st.warning("Enter a question first.")
            st.stop()

        with st.spinner("Running Single-Agent, Multi-Agent, and Uncertainty/Abstention independently…"):
            backend = _cached_backend(backend_name)
            # Run all three architectures
            comparison = run_live_comparison(
                question.strip(),
                question_id=question_id,
                question_source=question_source,
                reference_answer=reference_answer,
                backend=backend,
                backend_name=backend_name,
                run_id=create_run_id("phase20"),
            )

        payload = comparison.to_dict()
        devices = {((payload.get("results") or {}).get(name) or {}).get("device") for name in LIVE_ARCHITECTURES}
        if "mps_capable_host" in devices:
            st.error(
                "device=mps_capable_host means this browser hit the Mac Streamlit process "
                "(127.0.0.1:8501), not Colab T4. Close this tab and open the Colab proxy URL."
            )
            st.stop()
        st.session_state["live_comparison"] = payload
        if save_raw:
            path = _save_live_result(comparison)
            st.caption(f"Saved raw comparison to `{path}`.")

    payload = st.session_state.get("live_comparison")
    if not payload:
        st.caption("No live run yet.")
        return

    st.divider()
    st.markdown(f"**Run:** `{payload.get('run_id')}` · **question_id:** `{payload.get('question_id')}` · **source:** {payload.get('question_source')}")
    st.markdown(f"**Question:** {payload.get('question')}")
    if payload.get("error"):
        st.error(f"Live comparison error: {payload.get('error')}")

    cols = st.columns(3)
    results = payload.get("results") or {}
    for col, architecture in zip(cols, LIVE_ARCHITECTURES, strict=True):
        with col:
            data = results.get(architecture)
            if not data:
                st.warning(f"Missing result for {architecture}")
                continue
            # Show each architecture
            render_architecture(RAGCaseResult(**data))

    st.divider()
    st.subheader("Side-by-side summary")
    rows = []
    for architecture in LIVE_ARCHITECTURES:
        data = results.get(architecture) or {}
        verify = data.get("verification_result") or {}
        failed = data.get("decision") in LIVE_FAILURE_DECISIONS or data.get("error")
        rows.append(
            {
                "Architecture": ARCHITECTURE_LABELS.get(architecture, architecture),
                "Decision": data.get("decision"),
                "Confidence": "n/a" if failed else format_confidence_display(RAGCaseResult(**data)),
                "Threshold": format_threshold_display(RAGCaseResult(**data)),
                "Verification": None if failed else verify.get("verification_score"),
                "n_evidence": len(data.get("retrieved_evidence") or []),
                "Latency (s)": data.get("latency_seconds"),
                "Error": data.get("error"),
            }
        )
    st.dataframe(rows, use_container_width=True, hide_index=True)


def main() -> None:
    st.set_page_config(page_title="V2 RAG Artefact", layout="wide")
    apply_pending_app_page(st.session_state)
    apply_catalogue_prefill_to_live_input(st.session_state)
    with st.sidebar:
        st.radio(
            "Navigate",
            options=["Live RAG Demo", "Benchmark Results", "Benchmark Questions"],
            key="app_page",
        )
    page = st.session_state.get("app_page") or "Live RAG Demo"
    if page == "Benchmark Questions":
        render_benchmark_questions_page()
        return
    if page == "Benchmark Results":
        render_benchmark_results_page()
        return


In [ ]:
    render_live_rag_demo()


if __name__ == "__main__":
    main()


# 8. RQ / Results Connection

Very short map from research question to the real code. This is not a results write-up.

| RQ | What I point to | File / function |
|---|---|---|
| RQ1 | Correctness + paired comparison of Single-Agent vs Multi-Agent | `numeric_match()` / `answer_correctness` in `metrics.py`; confirmatory `mcnemar_exact(sa_disp, ma_disp)` in `analyse()` (`rq1_mcnemar_displayed_sa_vs_ma`) |
| RQ2 | Confidence vs judge faithfulness, and ANSWER vs ABSTAIN | `parsed_faithfulness_score` in `judge.py`; `spearman_corr(uq_conf, uq_llm)` and `mannwhitney(...)` in `analyse()` |
| RQ3 | Unsupported-emitted and ANSWER / ABSTAIN behaviour at T = 0.65 | `unsupported_emitted` in `score_case()`; coverage / selective accuracy in `aggregate_architecture()`; `mcnemar_exact` on unsupported flags in `analyse()` |

Holm is applied inside each RQ family by `_annotate_family()` → `holm_adjust()`.

The judge used for RQ2 is the **custom Qwen3-8B LLM-as-judge / RAGAS-inspired judge**, not official RAGAS.


# Professor asks → file to open

| Professor asks | Open this file | Look at this name |
|---|---|---|
| Where is retrieval? | `V2/src/retrieval/retriever.py` | `retrieve()` |
| Where are documents collected? | `V2/src/retrieval/pdf_fetch.py` | `collect_corpus_targets()`, `download_pdfs()` |
| Where is PDF extraction? | `V2/src/retrieval/extract.py` | `extract_pdf_pages()` |
| Where is chunking? | `V2/src/retrieval/chunking.py` | `split_text()`, `chunk_pages()` |
| Where are embeddings? | `V2/src/retrieval/embeddings.py` | `embed_texts()` |
| Where is ChromaDB? | `V2/src/retrieval/index.py` | `_chroma_client()`, `build_knowledge_base()`, `collection.add()` |
| Where is the vector query? | `V2/src/retrieval/retriever.py` | `collection.query` inside `retrieve()` |
| Where is top-k? | `V2/config/experiment.yaml` and `retriever.py` | `retrieval.top_k: 4` |
| Where is Single-Agent? | `V2/src/rag/single_agent.py` | `run_single_agent()` |
| Where is Multi-Agent? | `V2/src/rag/multi_agent.py` | `run_multi_agent()` |
| Where is verification? | `V2/src/rag/verification.py` | `compute_verification_result()` |
| Where is UQ? | `V2/src/rag/uncertainty.py` and `multi_agent_uq.py` | `compute_combined_confidence()`, `run_multi_agent_uq()` |
| Where is confidence? | `V2/src/rag/uncertainty.py` | `compute_combined_confidence()` |
| Where is the threshold? | `V2/src/calibration/select.py`, `lock.py`, `results/config/threshold.lock.json` | `select_threshold()`, locked **T = 0.65** |
| Where did 0.65 come from? | `V2/src/calibration/select.py` and `data.py` | `select_threshold()`, `COVERAGE_FLOOR`, DEV 40 |
| Where are prompts? | `V2/src/rag/prompts.py` and `V2/config/prompts.yaml` | `build_baseline_prompt()`, `build_multi_agent_verification_prompt()` |
| Where is evaluation? | `V2/src/evaluation/numeric.py`, `metrics.py`, `judge.py` | `numeric_match()`, `score_case()`, `judge_one_case()` |
| Where is the judge? | `V2/src/evaluation/judge.py` | `judge_one_case()`, `parsed_faithfulness_score` |
| Where is McNemar? | `V2/src/statistics/tests.py` | `mcnemar_exact()` |
| Where is Spearman? | `V2/src/statistics/tests.py` | `spearman_corr()` |
| Where is Mann–Whitney? | `V2/src/statistics/tests.py` | `mannwhitney()` |
| Where is Holm? | `V2/src/statistics/tests.py` | `holm_adjust()` |
| Where is Streamlit? | `V2/app/streamlit_app.py` | `main()`, `render_architecture()`, `render_live_rag_demo()` |
| What is the 0.66 warning? | `V2/src/rag/live.py` | `uq_ui_confidence_overlay()` — display only |
